# BRCA1 ClinProtGym ESM-C SAE downstream analysis

This notebook is for the `MV_BRCA1_Findlay_2018` rerun using the pooled/synonymous-corrected data and sequence-deduplicated SAE training. It keeps job submission, label swapping, AUC refreshes, LLR baseline generation, benchmark comparisons, gamma=1 SAE ensemble collection, and plotting in one place.

The job-submission cells default to dry-run-style behavior. Flip the `SUBMIT_*` or `RUN_*` switches in the setup cell before running them.

In [ ]:
from pathlib import Path
import json
import os
import pickle
import subprocess
import sys
import time

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def ensure_plotting():
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
    return plt, sns

PYTHON = str(Path(sys.executable).resolve())
PIPELINE = REPO_ROOT / "scripts" / "clinprotgym_esmc_sae_pipeline.py"
REFRESH_AUC = REPO_ROOT / "scripts" / "refresh_clinprotgym_clinvar_auc.py"
OUTPUT_ROOT = REPO_ROOT / "data" / "clinprotgym_esmc_sae"
DATASET = "MV_BRCA1_Findlay_2018"
RUN_LABEL = "DeltaEmbSAE_max_pool_batchtopk_k64_nf12800_seed42_seqdedup_v2"
DATASET_ROOT = OUTPUT_ROOT / "datasets" / DATASET
TABLE_DIR = DATASET_ROOT / "tables"
JOB_DIR = DATASET_ROOT / "jobs"
FIGURE_DIR = DATASET_ROOT / "figures" / "brca1_sae_downstream"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Label controls. Use "pipeline" for the labels in the prepared dataset state,
# "hgvs" for exact-HGVS labels via refresh_clinprotgym_clinvar_auc.py outputs,
# or "custom_csv" for a CSV with SequenceIndex, annotation, and optional stars columns.
CLINICAL_LABEL_MODE = "pipeline"  # "pipeline", "hgvs", or "custom_csv"
CUSTOM_CLINICAL_LABELS_CSV = None
ANNOTATION_SCHEME_FOR_REFRESH = "auto"  # "auto", "hgvs", or "protein"
FORCE_REBUILD_HGVS_CACHE = False

# Job controls. Keep submit/run switches False until you actually want to launch work.
SUBMIT_LLR_JOBS = False
RUN_LLR_LOCALLY = False
SUBMIT_BENCHMARK_JOBS = False
SUBMIT_ENSEMBLE_JOBS = False
RUN_COLLECT_SAE = False
RUN_COLLECT_BENCHMARKS = False
RUN_COLLECT_ENSEMBLES = False
RUN_REFRESH_AUC = False
RUN_SUMMARIZE = True

LLR_MEM = "64G"
LLR_TIME = "12:00:00"
LLR_MAX_ACTIVE = 2
BENCHMARK_MEM = "24G"
BENCHMARK_TIME = "00:30:00"
BENCHMARK_MAX_ACTIVE = 12
BENCHMARK_FORCE_RECOMPUTE = False
ENSEMBLE_MEM = "96G"
ENSEMBLE_TIME = "18:00:00"
ENSEMBLE_TOP_N = 12
ENSEMBLE_GAMMA = 1.0

REVIEW_STAR_CUTOFFS = [0, 1, 2, 3, 4]
DOWNSTREAM_PATCH_VERSION = "review-star-label-fix-v4"
PATHOGENICITY_LABELS = {"benign", "pathogenic"}
METHOD_ORDER = [
    "Enrichment ratio baseline",
    "popDMS baseline",
    "LLR baseline",
    "Raw embeddings",
    "Raw SAE",
    "Ensemble SAE model",
]
REFERENCE_METHODS = ["DMS functional score"]
METHOD_LABELS = {
    "DMS functional score": "DMS score",
    "Enrichment ratio baseline": "Enrichment ratio",
    "popDMS baseline": "popDMS",
    "LLR baseline": "LLR",
    "Raw embeddings": "Raw ESM-C",
    "Raw SAE": "DeltaEmbSAE single layer",
    "Ensemble SAE model": "DeltaEmbSAE ensemble",
}

def sh(cmd, *, run=True, check=True, capture=False):
    cmd = [str(x) for x in cmd]
    print(" ".join(cmd))
    if not run:
        return None
    completed = subprocess.run(cmd, check=check, text=True, capture_output=capture)
    if capture and completed.stdout:
        print(completed.stdout)
    if capture and completed.stderr:
        print(completed.stderr)
    return completed

def read_csv(path, **kwargs):
    path = Path(path)
    return pd.read_csv(path, **kwargs) if path.is_file() else pd.DataFrame()

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def clean_annotation_series(series):
    return series.astype(str).str.lower().replace({"nan": np.nan, "none": np.nan, "": np.nan})

def coalesce_first_present(df, candidate_columns, default=np.nan):
    # Seed with NaN, not `default`: combine_first only fills nulls, so a non-null default
    # (e.g. stars=0) would win over every real value and silently zero the column.
    out = pd.Series(np.nan, index=df.index, dtype=object)
    for column in candidate_columns:
        if column in df.columns:
            values = df[column]
            if isinstance(values, pd.DataFrame):
                values = values.iloc[:, 0]
            out = out.combine_first(values)
    return out.fillna(default)

def normalize_label_columns(df):
    df = df.copy()
    annotation_candidates = [
        "annotation_label", "annotation_active_label", "annotation_y",
        "clinvar_annotation", "annotation", "annotation_x", "annotation_fitness",
    ]
    star_candidates = [
        "stars_label", "stars_active_label", "stars_y", "clinvar_review_stars",
        "review_stars", "stars", "stars_x", "stars_fitness",
    ]
    df["annotation"] = clean_annotation_series(coalesce_first_present(df, annotation_candidates))
    df["stars"] = safe_numeric(coalesce_first_present(df, star_candidates, default=0)).fillna(0).astype(int)
    return df

print(f"Notebook patch: {DOWNSTREAM_PATCH_VERSION}")
print(f"Repo: {REPO_ROOT}")
print(f"Python: {PYTHON}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Figures: {FIGURE_DIR}")


## SAE Job Completion Audit

This checks the BRCA1 SAE task result files for the new sequence-deduplicated run label, verifies required artifacts exist, and scans the replacement Slurm logs for obvious failures.

In [ ]:
def audit_sae_results(dataset=DATASET, run_label=RUN_LABEL, slurm_job_id="56869892"):
    root = OUTPUT_ROOT / "datasets" / dataset / "jobs" / "fixed_sae_model_layer_array"
    rows = []
    for path in sorted(root.glob("*/Layer_*/task_result.json")):
        result = json.loads(path.read_text())
        model = path.parent.parent.name
        layer = int(path.parent.name.replace("Layer_", ""))
        required_paths = ["feature_path", "model_path", "viz_path"]
        if result.get("inference_path"):
            required_paths.append("inference_path")
        rows.append({
            "model": model,
            "layer": layer,
            "status": result.get("status"),
            "run_label": result.get("run_label"),
            "all_artifacts_exist": all(Path(result.get(key, "")).is_file() for key in required_paths),
            "feature_path": result.get("feature_path", ""),
            "model_path": result.get("model_path", ""),
            "viz_path": result.get("viz_path", ""),
            "inference_path": result.get("inference_path", ""),
            "error": result.get("error", ""),
        })
    result_df = pd.DataFrame(rows).sort_values(["model", "layer"]).reset_index(drop=True)
    current_df = result_df[result_df["run_label"].eq(run_label)].copy()

    log_dir = OUTPUT_ROOT / "jobs" / "fixed_deltaembsae_layer_array" / "logs"
    error_rows = []
    failure_tokens = ["Traceback", "ModuleNotFoundError", "No module named", "No space", "MemoryError", "Killed", "RuntimeError", "CANCELLED", "FAILED", "TIMEOUT"]
    for err_path in sorted(log_dir.glob(f"slurm-{slurm_job_id}_*.err")):
        text = err_path.read_text(errors="replace")
        hits = [token for token in failure_tokens if token in text]
        if hits or text.strip():
            error_rows.append({"log": str(err_path), "bytes": err_path.stat().st_size, "hits": ", ".join(hits), "preview": text[:500]})
    error_df = pd.DataFrame(error_rows)

    summary = pd.DataFrame([{
        "dataset": dataset,
        "expected_run_label": run_label,
        "result_files_all_labels": len(result_df),
        "result_files_current_label": len(current_df),
        "ok_current_label": int((current_df["status"].eq("ok") & current_df["all_artifacts_exist"]).sum()) if not current_df.empty else 0,
        "bad_current_label": int((~(current_df["status"].eq("ok") & current_df["all_artifacts_exist"])).sum()) if not current_df.empty else 0,
        "stderr_files_with_content_or_fail_tokens": len(error_df),
    }])
    return summary, current_df, error_df

sae_audit_summary, sae_result_df, sae_error_logs = audit_sae_results()
display(sae_audit_summary)
display(sae_result_df.groupby("model").agg(n_layers=("layer", "count"), min_layer=("layer", "min"), max_layer=("layer", "max"), ok=("status", lambda s: int(s.eq("ok").sum()))).reset_index())
if sae_error_logs.empty:
    print("No stderr failures found for the replacement SAE array logs.")
else:
    display(sae_error_logs)


## Collect SAE Metrics

Run this after the SAE array finishes. This writes the BRCA1 per-layer SAE metrics table and refreshes the global fixed-SAE metrics table from collected task outputs.

In [ ]:
collect_sae_cmd = [PYTHON, PIPELINE, "collect-sae", "--datasets", DATASET]
sh(collect_sae_cmd, run=RUN_COLLECT_SAE)

sae_metrics = read_csv(TABLE_DIR / f"{DATASET}_fixed_deltaembsae_layer_metrics.csv")
print(f"SAE metrics rows: {len(sae_metrics)}")
if not sae_metrics.empty:
    display(
        sae_metrics[["dataset", "model_short", "layer_index", "status", "run_label", "reconstruction_r2", "feature_path", "inference_path"]]
        .sort_values(["model_short", "layer_index"])
        .head(10)
    )


## Clinical Label Swapping And AUC Refresh

Use this section when only ClinVar/clinical labels changed. It does not rerun embeddings, SAE training, LLR, benchmarks, or ensembles. The first cell can call the pipeline refresh script for `auto`, `hgvs`, or `protein` annotation schemes. The second cell loads whichever labels you want to use for notebook-side AUC recalculation.

In [ ]:
refresh_auc_cmd = [PYTHON, REFRESH_AUC, "--datasets", DATASET, "--annotation-scheme", ANNOTATION_SCHEME_FOR_REFRESH, "--skip-prepare"]
if FORCE_REBUILD_HGVS_CACHE:
    refresh_auc_cmd.append("--force-rebuild-hgvs-cache")
sh(refresh_auc_cmd, run=RUN_REFRESH_AUC)


In [ ]:
def load_dataset_state(dataset=DATASET):
    state_path = OUTPUT_ROOT / "datasets" / dataset / "sequence_data" / f"{dataset}_processed_state.pkl"
    if not state_path.is_file():
        candidates = sorted((OUTPUT_ROOT / "datasets" / dataset).glob("**/*state*.pkl"))
        if candidates:
            state_path = candidates[0]
        else:
            raise FileNotFoundError(f"Missing processed state pickle for {dataset} under {OUTPUT_ROOT / 'datasets' / dataset}")
    with state_path.open("rb") as handle:
        return pickle.load(handle)

def load_pipeline_labels(dataset=DATASET):
    state = load_dataset_state(dataset)
    labels = state["annotations_dataframe"].copy()
    labels["SequenceIndex"] = labels["SequenceIndex"].astype(str)
    labels["annotation"] = clean_annotation_series(labels["annotation"])
    if "stars" not in labels.columns:
        labels["stars"] = 0
    labels["stars"] = safe_numeric(labels["stars"]).fillna(0).astype(int)
    return labels[["SequenceIndex", "annotation", "stars"]].drop_duplicates("SequenceIndex", keep="first")

def load_hgvs_labels(dataset=DATASET):
    paths = [
        OUTPUT_ROOT / "tables" / "hgvs_clinvar_annotations" / f"{dataset}_hgvs_clinvar_annotations.csv",
        TABLE_DIR / f"{dataset}_hgvs_clinvar_annotations.csv",
    ]
    path = next((p for p in paths if p.is_file()), None)
    if path is None:
        raise FileNotFoundError("No HGVS label cache found. Run the refresh AUC cell with ANNOTATION_SCHEME_FOR_REFRESH='hgvs' or 'auto'.")
    labels = pd.read_csv(path, dtype={"hgvs_nt": str})
    labels = labels.rename(columns={"hgvs_nt": "SequenceIndex", "clinvar_review_stars": "stars"})
    if "stars" not in labels.columns:
        labels["stars"] = 0
    labels["annotation"] = clean_annotation_series(labels["annotation"])
    labels["stars"] = safe_numeric(labels["stars"]).fillna(0).astype(int)
    return labels[["SequenceIndex", "annotation", "stars"]].drop_duplicates("SequenceIndex", keep="first")

def load_custom_labels(path):
    labels = pd.read_csv(path)
    if "SequenceIndex" not in labels.columns:
        for candidate in ["hgvs_nt", "mutant", "protein_sequence_index"]:
            if candidate in labels.columns:
                labels = labels.rename(columns={candidate: "SequenceIndex"})
                break
    if "SequenceIndex" not in labels.columns or "annotation" not in labels.columns:
        raise ValueError("Custom labels need SequenceIndex (or hgvs_nt/mutant/protein_sequence_index) and annotation columns.")
    if "stars" not in labels.columns:
        star_col = next((c for c in ["clinvar_review_stars", "review_stars", "star"] if c in labels.columns), None)
        labels["stars"] = labels[star_col] if star_col else 0
    labels["SequenceIndex"] = labels["SequenceIndex"].astype(str)
    labels["annotation"] = clean_annotation_series(labels["annotation"])
    labels["stars"] = safe_numeric(labels["stars"]).fillna(0).astype(int)
    return labels[["SequenceIndex", "annotation", "stars"]].drop_duplicates("SequenceIndex", keep="first")

def load_active_labels(mode=CLINICAL_LABEL_MODE):
    if mode == "pipeline":
        return load_pipeline_labels()
    if mode == "hgvs":
        return load_hgvs_labels()
    if mode == "custom_csv":
        if not CUSTOM_CLINICAL_LABELS_CSV:
            raise ValueError("Set CUSTOM_CLINICAL_LABELS_CSV before using custom_csv mode.")
        return load_custom_labels(CUSTOM_CLINICAL_LABELS_CSV)
    raise ValueError(f"Unknown CLINICAL_LABEL_MODE: {mode}")

active_labels = load_active_labels()
print(f"Loaded {len(active_labels)} labels from mode={CLINICAL_LABEL_MODE!r}")
active_labels = normalize_label_columns(active_labels)
display(active_labels["annotation"].value_counts(dropna=False).rename_axis("annotation").reset_index(name="n"))
display(active_labels.groupby("stars")["annotation"].value_counts().rename("n").reset_index().sort_values(["stars", "annotation"]))


## LLR Baseline Jobs

This creates masked-marginal ESM-C LLR jobs for BRCA1. If `RUN_LLR_LOCALLY=True`, it runs the payload tasks directly in the notebook kernel one at a time. Otherwise it writes a Slurm array and submits only if `SUBMIT_LLR_JOBS=True`.

The benchmark job creation cell below automatically includes LLR rows once the `*_llr_fitness.csv` files exist.

In [ ]:
llr_cmd = [
    PYTHON, PIPELINE, "create-llr-jobs",
    "--datasets", DATASET,
    "--python-executable", PYTHON,
    "--llr-mem", LLR_MEM,
    "--llr-time", LLR_TIME,
    "--max-active-llr-tasks", str(LLR_MAX_ACTIVE),
]
if SUBMIT_LLR_JOBS and not RUN_LLR_LOCALLY:
    llr_cmd.append("--submit")
sh(llr_cmd, run=True)

llr_payload = OUTPUT_ROOT / "jobs" / "llr" / "clinprotgym_llr_payload.pkl"
llr_task_table = read_csv(OUTPUT_ROOT / "tables" / "clinprotgym_llr_tasks.csv")
if not llr_task_table.empty:
    display(llr_task_table[llr_task_table["dataset"].eq(DATASET)])

if RUN_LLR_LOCALLY:
    with llr_payload.open("rb") as handle:
        payload = pickle.load(handle)
    for task_idx, task in enumerate(payload["tasks"]):
        if task["dataset"] != DATASET:
            continue
        sh([PYTHON, PIPELINE, "run-llr-task", llr_payload, str(task_idx)], run=True)


## Inspect LLR Outputs

Run this after the LLR Slurm jobs finish. It does not create or submit jobs.

In [ ]:
llr_task_table = read_csv(OUTPUT_ROOT / "tables" / "clinprotgym_llr_tasks.csv")
llr_status_dir = OUTPUT_ROOT / "jobs" / "llr" / "status"
llr_status_rows = []
for status_path in sorted(llr_status_dir.glob("*.json")):
    try:
        row = json.loads(status_path.read_text())
    except Exception as exc:
        row = {"status_path": str(status_path), "status": "unreadable", "error": repr(exc)}
    row["status_path"] = str(status_path)
    llr_status_rows.append(row)
llr_status = pd.DataFrame(llr_status_rows)
if not llr_status.empty:
    llr_status = llr_status[llr_status["dataset"].astype(str).eq(DATASET)].copy()
    display(llr_status)
else:
    print("No LLR status files found yet.")

llr_outputs = sorted(TABLE_DIR.glob(f"{DATASET}_*_llr_fitness.csv"))
print("LLR fitness outputs:")
for output_path in llr_outputs:
    print(output_path)


## Benchmark Baseline Comparison Jobs

This writes/submits the baseline comparison array for DMS score, enrichment ratio, popDMS, optional LLR, raw ESM-C layer features, and fixed SAE layer features. Run `collect-benchmarks` after the jobs finish.

In [ ]:
benchmark_cmd = [
    PYTHON, PIPELINE, "create-benchmark-jobs",
    "--datasets", DATASET,
    "--python-executable", PYTHON,
    "--benchmark-mem", BENCHMARK_MEM,
    "--benchmark-time", BENCHMARK_TIME,
    "--max-active-benchmark-tasks", str(BENCHMARK_MAX_ACTIVE),
]
if BENCHMARK_FORCE_RECOMPUTE:
    benchmark_cmd.append("--force-recompute")
if SUBMIT_BENCHMARK_JOBS:
    benchmark_cmd.append("--submit")
sh(benchmark_cmd, run=True)

benchmark_task_table = read_csv(OUTPUT_ROOT / "tables" / "clinprotgym_benchmark_tasks.csv")
if not benchmark_task_table.empty:
    brca1_tasks = benchmark_task_table[benchmark_task_table["dataset"].eq(DATASET)].copy()
    display(brca1_tasks["task_type"].value_counts().rename_axis("task_type").reset_index(name="n"))
    display(brca1_tasks.head())


In [ ]:
#sh(["sbatch", "--array=71-138%12", OUTPUT_ROOT / "jobs" / "benchmark_row_analysis" / "submit_clinprotgym_benchmark_row_analysis_array.sh"], run=True)

In [ ]:
#benchmark_metrics = read_csv(TABLE_DIR / f"{DATASET}_method_metrics.csv")
#benchmark_metrics["benchmark"].value_counts()

## Collect Benchmark Metrics

Run this after the benchmark Slurm array finishes. This only reads finished result CSVs and writes the collected metrics tables; it does not submit or recompute benchmark jobs.

In [ ]:
collect_benchmarks_cmd = [PYTHON, PIPELINE, "collect-benchmarks", "--datasets", DATASET]
sh(collect_benchmarks_cmd, run=RUN_COLLECT_BENCHMARKS)

benchmark_metrics = read_csv(TABLE_DIR / f"{DATASET}_method_metrics.csv")
print(f"Benchmark metric rows: {len(benchmark_metrics)}")
if not benchmark_metrics.empty:
    display(
        benchmark_metrics[[c for c in ["method_family", "benchmark", "model_label", "model_short", "layer", "spearman_rho", "auc", "fitness_path"] if c in benchmark_metrics.columns]]
        .head(12)
    )


## Gamma=1 SAE Ensemble Jobs

This uses the collected benchmark table to select the top fixed-SAE rows by Spearman/AUC and submit the rank-worst SAE ensemble at `gamma=1.0`. The output becomes another method family in the downstream plots.

In [ ]:
ensemble_cmd = [
    PYTHON, PIPELINE, "create-ensemble-jobs",
    "--datasets", DATASET,
    "--python-executable", PYTHON,
    "--ensemble-top-n", str(ENSEMBLE_TOP_N),
    "--ensemble-gamma", str(ENSEMBLE_GAMMA),
    "--ensemble-mem", ENSEMBLE_MEM,
    "--ensemble-time", ENSEMBLE_TIME,
]
if SUBMIT_ENSEMBLE_JOBS:
    ensemble_cmd.append("--submit")
    sh(ensemble_cmd, run=True)

    ensemble_task_table = read_csv(OUTPUT_ROOT / "tables" / "clinprotgym_sae_ensemble_tasks.csv")
    if not ensemble_task_table.empty:
        display(ensemble_task_table[ensemble_task_table["dataset"].eq(DATASET)])


## Collect SAE Ensemble Metrics

Run this after the ensemble Slurm job finishes. This only collects existing ensemble metrics; it does not create or submit ensemble jobs.

In [ ]:
collect_ensembles_cmd = [PYTHON, PIPELINE, "collect-ensembles", "--datasets", DATASET]
sh(collect_ensembles_cmd, run=RUN_COLLECT_ENSEMBLES)

ensemble_metrics = read_csv(TABLE_DIR / f"{DATASET}_sae_ensemble_metrics.csv")
print(f"Ensemble metric rows: {len(ensemble_metrics)}")
if not ensemble_metrics.empty:
    display(
        ensemble_metrics[[c for c in ["method_family", "benchmark", "model_label", "spearman_rho", "auc", "gamma", "fitness_path"] if c in ensemble_metrics.columns]]
    )


## Redraw Pipeline Summary Plots

Run this after collecting benchmarks/ensembles or after refreshing labels. It redraws the pipeline-level summary figures for the active dataset selection.

In [ ]:
summarize_cmd = [PYTHON, PIPELINE, "summarize", "--datasets", DATASET]
sh(summarize_cmd, run=RUN_SUMMARIZE)


## Load Metrics And Recompute AUC With Active Labels

This section lets you swap clinical labels without rerunning models. It reads each method's `fitness_path`, joins the currently active labels, and recalculates ClinVar AUC at each review-star cutoff.

In [ ]:
def load_metric_tables(dataset=DATASET):
    frames = []

    summary_paths = [
        OUTPUT_ROOT / "tables" / "clinprotgym_method_metrics.csv",
        OUTPUT_ROOT / "tables" / "clinprotgym_sae_ensemble_metrics.csv",
        TABLE_DIR / f"{dataset}_method_metrics.csv",
        TABLE_DIR / f"{dataset}_sae_ensemble_metrics.csv",
    ]
    for path in summary_paths:
        df = read_csv(path)
        if not df.empty:
            df["source_table"] = str(path)
            df["source_priority"] = 10
            frames.append(df)

    result_dir = OUTPUT_ROOT / "datasets" / dataset / "jobs" / "benchmark_row_analysis" / "results"
    if result_dir.is_dir():
        for path in sorted(result_dir.glob("*.csv")):
            df = read_csv(path)
            if not df.empty:
                df["source_table"] = str(path)
                df["source_priority"] = 20
                frames.append(df)

    if not frames:
        return pd.DataFrame()

    metrics = pd.concat(frames, ignore_index=True, sort=False)
    if "dataset" in metrics.columns:
        metrics = metrics[metrics["dataset"].astype(str).eq(dataset)].copy()

    # Keep the current sequence-deduplicated DeltaEmbSAE run, not older same-index SAE result files.
    if "method_family" in metrics.columns:
        is_single_sae = metrics["method_family"].astype(str).eq("Raw SAE")
        run_text = pd.Series("", index=metrics.index, dtype=object)
        for col in ["model_label", "feature_path", "inference_path", "fitness_path", "source_table"]:
            if col in metrics.columns:
                run_text = run_text.str.cat(metrics[col].astype(str), sep=" ")
        active_single_sae = run_text.str.contains(RUN_LABEL, regex=False, na=False)
        metrics = metrics[~is_single_sae | active_single_sae].copy()

    for col in ["spearman_rho", "auc", "n_variants", "n_benign", "n_pathogenic", "layer", "gamma"]:
        if col in metrics.columns:
            metrics[col] = safe_numeric(metrics[col])

    dedupe_cols = [c for c in ["method_family", "benchmark", "model_label", "model", "layer", "fitness_path"] if c in metrics.columns]
    if dedupe_cols:
        metrics = metrics.sort_values("source_priority").drop_duplicates(dedupe_cols, keep="last")
    return metrics.reset_index(drop=True)

def auc_from_labeled_fitness(df):
    df = normalize_label_columns(df)
    if "fitness" not in df.columns:
        return {"auc": np.nan, "auc_discrimination": np.nan, "n_benign": 0, "n_pathogenic": 0, "n_variants": 0}
    df = df[df["annotation"].isin(PATHOGENICITY_LABELS)].dropna(subset=["fitness"]).copy()
    if df.empty:
        return {"auc": np.nan, "auc_discrimination": np.nan, "n_benign": 0, "n_pathogenic": 0, "n_variants": 0}
    scores = -safe_numeric(df["fitness"]).to_numpy(dtype=float)
    labels = df["annotation"].eq("pathogenic").to_numpy()
    finite = np.isfinite(scores)
    scores = scores[finite]
    labels = labels[finite]
    n_pos = int(labels.sum())
    n_neg = int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        auc = np.nan
    else:
        ranks = pd.Series(scores).rank(method="average").to_numpy()
        auc = float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
    return {
        "auc": auc,
        "auc_discrimination": max(auc, 1 - auc) if np.isfinite(auc) else np.nan,
        "n_benign": n_neg,
        "n_pathogenic": n_pos,
        "n_variants": int(len(labels)),
    }

def fitness_with_active_labels(fitness_path, min_stars=0, labels=active_labels):
    fitness_path = Path(str(fitness_path))
    if not fitness_path.is_file():
        return pd.DataFrame()
    fitness = pd.read_csv(fitness_path)
    if "SequenceIndex" not in fitness.columns or "fitness" not in fitness.columns:
        return pd.DataFrame()
    # The active label table is authoritative. A fitness CSV carries its own baked-in annotation
    # column, and leaving it in place lets normalize_label_columns fall back to it for variants the
    # star cutoff just dropped, which makes every cutoff a no-op.
    fitness = fitness.drop(columns=[c for c in ["annotation", "stars"] if c in fitness.columns])
    label_df = normalize_label_columns(labels.copy())
    label_df = label_df[label_df["stars"].ge(min_stars)].copy()
    label_df = label_df[label_df["annotation"].isin(PATHOGENICITY_LABELS)].copy()
    fitness["SequenceIndex"] = fitness["SequenceIndex"].astype(str)
    joined = fitness.merge(
        label_df[["SequenceIndex", "annotation", "stars"]],
        on="SequenceIndex",
        how="left",
        suffixes=("_fitness", "_label"),
    )
    joined = normalize_label_columns(joined)
    joined["fitness"] = safe_numeric(joined["fitness"])
    return joined

def recompute_auc_table(metrics, labels=active_labels, star_cutoffs=REVIEW_STAR_CUTOFFS):
    rows = []
    for idx, row in metrics.iterrows():
        fitness_path = row.get("fitness_path", "")
        if pd.isna(fitness_path) or not str(fitness_path):
            continue
        for min_stars in star_cutoffs:
            joined = fitness_with_active_labels(fitness_path, min_stars=min_stars, labels=labels)
            auc_stats = auc_from_labeled_fitness(joined)
            rows.append({
                **row.to_dict(),
                "metric_row_index": idx,
                "min_review_stars": min_stars,
                "review_cutoff": f">={min_stars} stars" if min_stars else "all binary",
                "label_mode": CLINICAL_LABEL_MODE,
                **auc_stats,
            })
    return pd.DataFrame(rows)

metrics = load_metric_tables()
auc_by_labels = recompute_auc_table(metrics)
print(f"Metric rows: {len(metrics)}")
if not metrics.empty and "method_family" in metrics.columns:
    display(metrics["method_family"].value_counts().rename_axis("method_family").reset_index(name="n"))
print(f"AUC rows with active labels: {len(auc_by_labels)}")
if not metrics.empty:
    display(metrics[[c for c in ["method_family", "benchmark", "model_label", "model_short", "layer", "spearman_rho", "auc", "fitness_path", "source_table"] if c in metrics.columns]].head(12))
if not auc_by_labels.empty:
    display(auc_by_labels[["method_family", "model_label", "min_review_stars", "auc", "auc_discrimination", "n_benign", "n_pathogenic"]].head(12))


## Best Rows By Method Family

For AUC-based clinical plots, the best row is selected by direction-normalized AUC (`max(AUC, 1-AUC)`) rather than Spearman.

In [ ]:
def method_label(value):
    return METHOD_LABELS.get(str(value), str(value))

def method_color(value):
    return {
        "DMS functional score": "#7F3C8D",
        "Enrichment ratio baseline": "#9D755D",
        "popDMS baseline": "#F58518",
        "LLR baseline": "#72B7B2",
        "Raw embeddings": "#4C78A8",
        "Raw SAE": "#54A24B",
        "Ensemble SAE model": "#E45756",
    }.get(str(value), "0.4")

def method_marker(value):
    return {
        "DMS functional score": "s",
        "Enrichment ratio baseline": "P",
        "popDMS baseline": "D",
        "LLR baseline": "v",
        "Raw embeddings": "o",
        "Raw SAE": "^",
        "Ensemble SAE model": "X",
    }.get(str(value), "o")

def row_key_tuple(row, key_cols):
    return tuple("" if pd.isna(row.get(col)) else str(row.get(col)) for col in key_cols)

def mark_best_rows(rows, best_rows, key_cols=("method_family", "model_label", "fitness_path")):
    if rows.empty or best_rows.empty:
        return pd.Series(False, index=rows.index)
    key_cols = [c for c in key_cols if c in rows.columns and c in best_rows.columns]
    if not key_cols:
        return pd.Series(False, index=rows.index)
    best_keys = {row_key_tuple(row, key_cols) for _, row in best_rows.iterrows()}
    return rows.apply(lambda row: row_key_tuple(row, key_cols) in best_keys, axis=1)

def best_rows_by_family(metrics, auc_table, min_stars=0):
    auc0 = auc_table[auc_table["min_review_stars"].eq(min_stars)].copy()
    if auc0.empty:
        return pd.DataFrame()
    auc0["spearman_rho"] = safe_numeric(auc0.get("spearman_rho", np.nan))
    auc0["auc_discrimination"] = safe_numeric(auc0["auc_discrimination"])
    rows = []
    for family in METHOD_ORDER:
        group = auc0[auc0["method_family"].eq(family)].copy()
        if group.empty:
            continue
        rows.append(group.sort_values(["auc_discrimination", "spearman_rho"], ascending=[False, False]).iloc[0])
    return pd.DataFrame(rows)

best_auc_rows = best_rows_by_family(metrics, auc_by_labels, min_stars=0)
if best_auc_rows.empty:
    print("No best rows yet. Collect benchmarks/ensembles first, or check that fitness_path files exist.")
else:
    best_auc_rows["method_plot_label"] = best_auc_rows["method_family"].map(method_label)
    display(best_auc_rows[["method_plot_label", "model_label", "spearman_rho", "auc", "auc_discrimination", "n_benign", "n_pathogenic", "fitness_path"]])


## Plot: Method Bars For BRCA1

Bar charts by method family for BRCA1, with Spearman rho and active-label ClinVar AUC side by side.

In [ ]:
plt, sns = ensure_plotting()
BAR_REVIEW_STAR_CUTOFF = 0

if auc_by_labels.empty or best_auc_rows.empty:
    print("No rows to plot. Run the metrics/AUC recompute cells first.")
else:
    all_rows = auc_by_labels[auc_by_labels["min_review_stars"].eq(BAR_REVIEW_STAR_CUTOFF)].copy()
    all_rows["spearman_rho"] = safe_numeric(all_rows.get("spearman_rho", np.nan))
    all_rows["auc_discrimination"] = safe_numeric(all_rows["auc_discrimination"])
    all_rows["method_plot_label"] = all_rows["method_family"].map(method_label)
    all_rows["is_best"] = mark_best_rows(all_rows, best_auc_rows)

    plot_df = best_auc_rows.copy()
    order = [method_label(m) for m in METHOD_ORDER if method_label(m) in set(all_rows["method_plot_label"])]
    x_lookup = {label: idx for idx, label in enumerate(order)}

    fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.1), sharex=False)
    metric_specs = [
        ("spearman_rho", "Spearman rho", "BRCA1 DMS agreement", 0, None),
        ("auc_discrimination", "Direction-normalized ClinVar AUC", f"BRCA1 clinical labels: {CLINICAL_LABEL_MODE}", 0.5, (0, 1.02)),
    ]

    for ax, (metric, ylabel, title, refline, ylim) in zip(axes, metric_specs):
        sns.barplot(
            data=plot_df,
            x="method_plot_label",
            y=metric,
            order=order,
            ax=ax,
            color="0.86",
            edgecolor="0.35",
            linewidth=0.8,
        )
        for label in order:
            sub = all_rows[all_rows["method_plot_label"].eq(label)].copy()
            sub = sub[np.isfinite(sub[metric])]
            if sub.empty:
                continue
            xs = np.full(len(sub), x_lookup[label], dtype=float)
            if len(sub) > 1:
                xs += np.linspace(-0.24, 0.24, len(sub))
            colors = [method_color(v) for v in sub["method_family"]]
            nonbest = ~sub["is_best"].to_numpy(dtype=bool)
            ax.scatter(xs[nonbest], sub.loc[nonbest, metric], s=34, c=np.asarray(colors, dtype=object)[nonbest], alpha=0.20, edgecolors="none", zorder=3)
            best = sub["is_best"].to_numpy(dtype=bool)
            ax.scatter(xs[best], sub.loc[best, metric], s=112, c=np.asarray(colors, dtype=object)[best], alpha=0.98, edgecolors="black", linewidths=0.8, zorder=5)
        ax.axhline(refline, color="0.65", linewidth=1, linestyle="--" if metric == "auc_discrimination" else "-")
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.set_xlabel("Method")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order, rotation=35, ha="right")

    fig.suptitle("Bars show the best row per method; translucent points show all rows/layers", y=1.03)
    fig.tight_layout()
    out = FIGURE_DIR / f"{DATASET}_best_method_spearman_auc_bars_{CLINICAL_LABEL_MODE}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(out)


## Plot: AUC By ClinVar Review-Star Cutoff

This mirrors the review-star cutoff plots from the ensemble notebook, but uses the active clinical label table and selects best models by AUC.

In [ ]:
plt, sns = ensure_plotting()
if auc_by_labels.empty:
    print("No AUC rows to plot.")
else:
    plot_df = auc_by_labels.copy()
    plot_df["auc_discrimination"] = safe_numeric(plot_df["auc_discrimination"])
    plot_df["method_plot_label"] = plot_df["method_family"].map(method_label)
    plot_df["is_best"] = mark_best_rows(plot_df, best_auc_rows)
    plot_df = plot_df[np.isfinite(plot_df["auc_discrimination"])].copy()

    if plot_df.empty:
        print("No finite AUC cutoff rows to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10.4, 5.6))
        seen_labels = set()
        for family in METHOD_ORDER:
            family_df = plot_df[plot_df["method_family"].eq(family)].copy()
            if family_df.empty:
                continue
            color = method_color(family)
            marker = method_marker(family)
            label_base = method_label(family)
            is_layer_family = family in {"Raw embeddings", "Raw SAE"}
            for _, group in family_df.groupby(["model_label", "fitness_path"], dropna=False):
                group = group.sort_values("min_review_stars")
                is_best = bool(group["is_best"].any())
                if is_layer_family and not is_best:
                    label = f"{label_base} all layers" if f"{label_base} all layers" not in seen_labels else "_nolegend_"
                    seen_labels.add(f"{label_base} all layers")
                    ax.plot(group["min_review_stars"], group["auc_discrimination"], color=color, alpha=0.13, linewidth=0.8, label=label, zorder=1)
                    ax.scatter(group["min_review_stars"], group["auc_discrimination"], color=color, alpha=0.13, s=18, marker=marker, edgecolors="none", zorder=1)
                else:
                    if is_best and is_layer_family:
                        label = f"Best {label_base}" if f"Best {label_base}" not in seen_labels else "_nolegend_"
                        seen_labels.add(f"Best {label_base}")
                    else:
                        label = label_base if label_base not in seen_labels else "_nolegend_"
                        seen_labels.add(label_base)
                    ax.plot(group["min_review_stars"], group["auc_discrimination"], color=color, alpha=0.96, linewidth=2.4 if is_best else 1.8, label=label, zorder=4 if is_best else 3)
                    ax.scatter(group["min_review_stars"], group["auc_discrimination"], color=color, alpha=0.98, s=72 if is_best else 48, marker=marker, edgecolors="black" if is_best else "white", linewidths=0.7 if is_best else 0.3, zorder=5 if is_best else 3)
        ax.axhline(0.5, color="0.7", linewidth=1, linestyle="--")
        ax.set_ylim(0, 1.02)
        ax.set_xticks(REVIEW_STAR_CUTOFFS)
        ax.set_xticklabels([f">={s} stars" if s else "all binary" for s in REVIEW_STAR_CUTOFFS])
        ax.set_xlabel("ClinVar review-star cutoff")
        ax.set_ylabel("Direction-normalized ClinVar AUC")
        ax.set_title(f"BRCA1 AUC by ClinVar review-star cutoff ({CLINICAL_LABEL_MODE})")
        ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_auc_by_review_star_cutoff_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)
        display(plot_df[plot_df["is_best"]][["method_plot_label", "model_label", "review_cutoff", "auc", "auc_discrimination", "n_benign", "n_pathogenic"]])


## Plot: Spearman Versus AUC

Scatter plot of Spearman rho vs active-label ClinVar AUC for every raw ESM-C layer and every DeltaEmbSAE layer. All layer points are transparent; the best raw ESM-C and best DeltaEmbSAE points are overlaid opaque. Ensemble and non-layer baselines are shown when available, with dotted AUC guide lines for LLR and DMS functional-score baselines.

In [ ]:
plt, sns = ensure_plotting()
SCATTER_REVIEW_STAR_CUTOFF = 0

if auc_by_labels.empty:
    print("No AUC rows to plot. Run the metrics/AUC recompute cells first.")
else:
    scatter_df = auc_by_labels[auc_by_labels["min_review_stars"].eq(SCATTER_REVIEW_STAR_CUTOFF)].copy()
    scatter_df["spearman_rho"] = safe_numeric(scatter_df.get("spearman_rho", np.nan))
    scatter_df["auc_discrimination"] = safe_numeric(scatter_df["auc_discrimination"])
    scatter_df["layer"] = safe_numeric(scatter_df.get("layer", np.nan))
    scatter_df["method_plot_label"] = scatter_df["method_family"].map(method_label)
    scatter_df["is_best"] = mark_best_rows(scatter_df, best_auc_rows)
    scatter_df = scatter_df[scatter_df["method_family"].isin(METHOD_ORDER)].copy()
    scatter_df = scatter_df[np.isfinite(scatter_df["spearman_rho"]) & np.isfinite(scatter_df["auc_discrimination"])].copy()

    layer_df = scatter_df[scatter_df["method_family"].isin(["Raw embeddings", "Raw SAE"])].copy()
    baseline_df = scatter_df[scatter_df["method_family"].isin(["Enrichment ratio baseline", "popDMS baseline", "LLR baseline"])].copy()
    ensemble_df = scatter_df[scatter_df["method_family"].eq("Ensemble SAE model")].copy()

    fig, ax = plt.subplots(figsize=(9.8, 6.5))

    for family, group in layer_df.groupby("method_family", sort=False):
        color = method_color(family)
        marker = method_marker(family)
        label_base = method_label(family)
        nonbest = group[~group["is_best"]].copy()
        if not nonbest.empty:
            ax.scatter(
                nonbest["spearman_rho"],
                nonbest["auc_discrimination"],
                s=46,
                marker=marker,
                color=color,
                alpha=0.18,
                edgecolor="none",
                label=f"{label_base} all layers",
                zorder=2,
            )
        # Points are identified in the legend rather than annotated in place; the best raw and best SAE
        # points land almost on top of each other, so on-plot labels overlapped.
        for _, row in group[group["is_best"]].iterrows():
            layer_text = "" if pd.isna(row.get("layer")) else f" L{int(row['layer'])}"
            model_text = str(row.get("model_short", "")).replace("nan", "")
            ax.scatter(
                row["spearman_rho"],
                row["auc_discrimination"],
                s=180,
                marker=marker,
                color=color,
                alpha=0.98,
                edgecolor="black",
                linewidth=0.9,
                label=f"Best {label_base}: {model_text}{layer_text} (AUC*={row['auc_discrimination']:.3f})",
                zorder=6,
            )

    for _, row in baseline_df.iterrows():
        family = row["method_family"]
        ax.scatter(
            row["spearman_rho"],
            row["auc_discrimination"],
            s=145,
            marker=method_marker(family),
            color=method_color(family),
            alpha=0.94,
            edgecolor="black",
            linewidth=0.8,
            label=f"{method_label(family)} (AUC*={row['auc_discrimination']:.3f})",
            zorder=5,
        )

    for _, row in ensemble_df.iterrows():
        family = row["method_family"]
        ax.scatter(
            row["spearman_rho"],
            row["auc_discrimination"],
            s=230,
            marker=method_marker(family),
            color=method_color(family),
            alpha=0.98,
            edgecolor="black",
            linewidth=1.0,
            label=f"{method_label(family)} gamma=1 (AUC*={row['auc_discrimination']:.3f})",
            zorder=8,
        )

    # LLR guide lines. The benchmark array does not always emit "LLR baseline" rows, so fall back to
    # scoring the masked-marginal LLR fitness tables directly against the active labels.
    def llr_guide_lines(min_stars=SCATTER_REVIEW_STAR_CUTOFF):
        rows = scatter_df[scatter_df["method_family"].eq("LLR baseline")].copy()
        if not rows.empty:
            return [
                (str(row.get("model_label") or row.get("model_short") or method_label("LLR baseline")), float(row["auc_discrimination"]))
                for _, row in rows.iterrows()
            ]
        guides = []
        for path in sorted(TABLE_DIR.glob(f"{DATASET}_*_llr_fitness.csv")):
            stats = auc_from_labeled_fitness(fitness_with_active_labels(path, min_stars=min_stars))
            if not np.isfinite(stats["auc_discrimination"]):
                continue
            model_tag = path.stem.replace(f"{DATASET}_", "").replace("_llr_fitness", "").split("__")[-1]
            guides.append((f"{method_label('LLR baseline')} {model_tag}", stats["auc_discrimination"]))
        return guides

    for label, y in llr_guide_lines():
        color = method_color("LLR baseline")
        ax.axhline(y, color=color, linestyle=":", linewidth=1.3, alpha=0.8, zorder=0)
        ax.text(0.01, y, f"{label} AUC*={y:.3f}", transform=ax.get_yaxis_transform(), va="bottom", ha="left", fontsize=8, color=color)

    # DMS functional score is a reference method, not a point in METHOD_ORDER, so it only appears as a guide line.
    reference_df = auc_by_labels[
        auc_by_labels["min_review_stars"].eq(SCATTER_REVIEW_STAR_CUTOFF)
        & auc_by_labels["method_family"].isin(REFERENCE_METHODS)
    ].copy()
    reference_df["auc_discrimination"] = safe_numeric(reference_df["auc_discrimination"])
    reference_df = (
        reference_df[np.isfinite(reference_df["auc_discrimination"])]
        .sort_values("auc_discrimination", ascending=False)
        .drop_duplicates("method_family")
    )
    for _, row in reference_df.iterrows():
        family = row["method_family"]
        y = row["auc_discrimination"]
        color = method_color(family)
        ax.axhline(y, color=color, linestyle="--", linewidth=1.4, alpha=0.8, zorder=0)
        ax.text(0.01, y, f"{method_label(family)} AUC*={y:.3f}", transform=ax.get_yaxis_transform(), va="bottom", ha="left", fontsize=8, color=color)

    ax.axvline(0, color="0.84", linewidth=1)
    ax.axhline(0.5, color="0.72", linewidth=1, linestyle="--")
    ax.set_ylim(0.45, 1.02)
    ax.set_xlabel("Spearman rho vs BRCA1 functional score")
    ax.set_ylabel("Direction-normalized ClinVar AUC")
    cutoff_label = "all binary labels" if SCATTER_REVIEW_STAR_CUTOFF == 0 else f">={SCATTER_REVIEW_STAR_CUTOFF} review stars"
    ax.set_title(f"BRCA1 individual layers and ensemble ({CLINICAL_LABEL_MODE}, {cutoff_label})")

    handles, labels = ax.get_legend_handles_labels()
    seen = set()
    dedup_handles, dedup_labels = [], []
    for handle, label in zip(handles, labels):
        if label in seen or label == "":
            continue
        seen.add(label)
        dedup_handles.append(handle)
        dedup_labels.append(label)
    ax.legend(dedup_handles, dedup_labels, title="Point set", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    fig.tight_layout()
    out = FIGURE_DIR / f"{DATASET}_all_layers_spearman_vs_auc_{CLINICAL_LABEL_MODE}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(out)
    display(scatter_df.groupby("method_plot_label").size().rename("n_points").reset_index())


## Plot: Layer-Wise Raw ESM-C And SAE Performance

This shows whether the new sequence-deduplicated SAE run changes the layer profile relative to raw max-pooled ESM-C embeddings.

In [ ]:
plt, sns = ensure_plotting()
LAYERWISE_REVIEW_STAR_CUTOFF = 0

if auc_by_labels.empty:
    print("No metrics loaded.")
else:
    layer_df = auc_by_labels[
        auc_by_labels["min_review_stars"].eq(LAYERWISE_REVIEW_STAR_CUTOFF)
        & auc_by_labels["method_family"].isin(["Raw embeddings", "Raw SAE"])
    ].copy()
    if layer_df.empty:
        print("No layer-wise raw embedding/DeltaEmbSAE rows. Run and collect benchmark jobs first.")
    else:
        layer_df["layer"] = safe_numeric(layer_df["layer"])
        layer_df["spearman_rho"] = safe_numeric(layer_df["spearman_rho"])
        layer_df["auc_discrimination"] = safe_numeric(layer_df["auc_discrimination"])
        layer_df["method_plot_label"] = layer_df["method_family"].map(method_label)
        layer_df["is_best"] = mark_best_rows(layer_df, best_auc_rows)

        ensemble_df = auc_by_labels[
            auc_by_labels["min_review_stars"].eq(LAYERWISE_REVIEW_STAR_CUTOFF)
            & auc_by_labels["method_family"].eq("Ensemble SAE model")
        ].copy()
        if not ensemble_df.empty:
            ensemble_df["spearman_rho"] = safe_numeric(ensemble_df["spearman_rho"])
            ensemble_df["auc_discrimination"] = safe_numeric(ensemble_df["auc_discrimination"])

        fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.3), sharex=False)
        metric_specs = [
            ("spearman_rho", "Spearman rho", "Layer-wise DMS agreement", 0),
            ("auc_discrimination", "Direction-normalized AUC", "Layer-wise clinical discrimination", 0.5),
        ]

        for ax, (metric, ylabel, title, refline) in zip(axes, metric_specs):
            for (family, model_short), group in layer_df.groupby(["method_family", "model_short"], dropna=False):
                group = group.sort_values("layer")
                color = method_color(family)
                marker = method_marker(family)
                label_base = f"{method_label(family)} {model_short}"
                ax.plot(group["layer"], group[metric], color=color, alpha=0.34, linewidth=1.25, label=label_base)
                ax.scatter(group["layer"], group[metric], color=color, alpha=0.22, s=24, marker=marker, edgecolors="none")

                best = group[group["is_best"]].copy()
                if not best.empty:
                    ax.scatter(best["layer"], best[metric], color=color, alpha=0.98, s=140, marker=marker, edgecolors="black", linewidths=0.9, label=f"Best {method_label(family)} {model_short}", zorder=6)

            if not ensemble_df.empty and metric in ensemble_df.columns:
                for _, row in ensemble_df.iterrows():
                    y = row.get(metric)
                    if pd.notna(y) and np.isfinite(y):
                        ax.axhline(y, color=method_color("Ensemble SAE model"), linestyle="--", linewidth=1.8, alpha=0.80, label=f"{method_label('Ensemble SAE model')} {metric}")
            ax.axhline(refline, color="0.8", linewidth=1, linestyle="--" if metric == "auc_discrimination" else "-")
            if metric == "auc_discrimination":
                ax.set_ylim(0, 1.02)
            ax.set_xlabel("ESM-C layer")
            ax.set_ylabel(ylabel)
            ax.set_title(title)
            handles, labels = ax.get_legend_handles_labels()
            seen = set()
            keep_h, keep_l = [], []
            for h, l in zip(handles, labels):
                if l in seen:
                    continue
                seen.add(l)
                keep_h.append(h)
                keep_l.append(l)
            ax.legend(keep_h, keep_l, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8)

        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_layerwise_raw_vs_sae_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)


## Plot: Best Method Fitness Distribution By Clinical Annotation

The best method is selected by active-label AUC, then its fitness values are split by benign/pathogenic labels across review-star cutoffs.

In [ ]:
plt, sns = ensure_plotting()
if best_auc_rows.empty:
    print("No best AUC row available.")
else:
    best_row = best_auc_rows.sort_values(["auc_discrimination", "spearman_rho"], ascending=[False, False]).iloc[0]
    best_fitness_path = Path(str(best_row["fitness_path"]))
    print(f"Best by AUC: {method_label(best_row['method_family'])} | {best_row['model_label']} | {best_fitness_path}")
    panels = []
    for min_stars in REVIEW_STAR_CUTOFFS:
        panel = fitness_with_active_labels(best_fitness_path, min_stars=min_stars)
        panel = normalize_label_columns(panel)
        panel = panel[panel["annotation"].isin(PATHOGENICITY_LABELS) & np.isfinite(panel["fitness"])].copy()
        if panel.empty:
            continue
        stats = auc_from_labeled_fitness(panel)
        panels.append((min_stars, panel, stats))
    if not panels:
        print("No labeled variants overlap this fitness table.")
    else:
        finite = pd.concat([p[1] for p in panels], ignore_index=True)["fitness"].to_numpy(dtype=float)
        bins = np.histogram_bin_edges(finite, bins=min(30, max(8, int(np.sqrt(len(finite))))))
        ncols = 2
        nrows = int(np.ceil(len(panels) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.2 * nrows), squeeze=False)
        palette = {"benign": "#2C7FB8", "pathogenic": "#D7301F"}
        for ax, (min_stars, panel, stats) in zip(axes.ravel(), panels):
            for annotation in ["benign", "pathogenic"]:
                values = panel.loc[panel["annotation"].eq(annotation), "fitness"].to_numpy(dtype=float)
                ax.hist(values, bins=bins, alpha=0.58, color=palette[annotation], label=f"{annotation} (n={len(values)})", edgecolor="white", linewidth=0.4)
            ax.set_title(f"{'>=' + str(min_stars) + ' stars' if min_stars else 'all binary'} | AUC*={stats['auc_discrimination']:.3f}")
            ax.set_xlabel("Fitness")
            ax.set_ylabel("Variant count")
            ax.legend(frameon=False)
        for ax in axes.ravel()[len(panels):]:
            ax.axis("off")
        fig.suptitle(f"BRCA1 best method fitness distributions: {method_label(best_row['method_family'])}", y=1.01)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_best_auc_method_fitness_distribution_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)


## SAE Ensemble Component Contributions

For the gamma=1 rank-worst ensemble, this counts which source SAE component supplies the selected worst rank for each variant. This mirrors the component-contribution analysis in the old ensemble notebook.

In [ ]:
plt, sns = ensure_plotting()
component_paths_table = TABLE_DIR / "sae_ensemble_gamma1" / f"{DATASET}_ensemble_component_fitness_paths.csv"
source_table = TABLE_DIR / "sae_ensemble_gamma1" / f"{DATASET}_ensemble_sources.csv"
components = read_csv(component_paths_table)
sources = read_csv(source_table)
if components.empty:
    print(f"No ensemble component table found yet: {component_paths_table}")
else:
    series = []
    for _, row in components.iterrows():
        path = Path(str(row["fitness_path"]))
        if not path.is_file():
            continue
        fitness = pd.read_csv(path)[["SequenceIndex", "fitness"]].copy()
        fitness["SequenceIndex"] = fitness["SequenceIndex"].astype(str)
        name = f"{row.get('model_short', row.get('model', 'model'))}_L{int(float(row['layer']))}"
        series.append(pd.Series(safe_numeric(fitness["fitness"]).to_numpy(dtype=float), index=fitness["SequenceIndex"], name=name))
    if not series:
        print("No component fitness files were readable.")
    else:
        score_matrix = pd.concat(series, axis=1)
        rank_matrix = score_matrix.rank(method="average", ascending=True)
        selected_component = rank_matrix.idxmin(axis=1)
        contribution = selected_component.value_counts(normalize=True).rename_axis("component").reset_index(name="fraction_selected")
        contribution["n_selected"] = selected_component.value_counts().reindex(contribution["component"]).to_numpy()
        if not sources.empty:
            sources = sources.copy()
            sources["component"] = sources.apply(lambda r: f"{r.get('model_short', r.get('model', 'model'))}_L{int(float(r['layer']))}", axis=1)
            contribution = contribution.merge(sources[["component", "spearman_rho", "auc", "model_label"]], on="component", how="left")
        display(contribution)
        fig, ax = plt.subplots(figsize=(10, 4.8))
        sns.barplot(data=contribution, x="component", y="fraction_selected", color="#54A24B", ax=ax)
        ax.set_xlabel("SAE component")
        ax.set_ylabel("Fraction selected by rank-worst ensemble")
        ax.set_title("BRCA1 gamma=1 SAE ensemble component contributions")
        ax.tick_params(axis="x", rotation=45)
        for tick in ax.get_xticklabels():
            tick.set_horizontalalignment("right")
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_sae_ensemble_component_contributions.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)


## UMAP/PCA Of SAE Ensemble Component Ranks

This uses the component-rank matrix from the gamma=1 ensemble. If `umap-learn` is not installed, it falls back to PCA so the cell still produces an embedding plot.

In [ ]:
plt, sns = ensure_plotting()
if "score_matrix" not in globals() or score_matrix.empty:
    print("Run the component contribution cell first, after ensemble outputs exist.")
else:
    rank_matrix = score_matrix.rank(method="average", ascending=True).dropna(how="any")
    if len(rank_matrix) < 3:
        print("Not enough complete variants for embedding.")
    else:
        try:
            import umap
            reducer = umap.UMAP(n_components=2, n_neighbors=min(30, len(rank_matrix) - 1), min_dist=0.12, random_state=42)
            coords = reducer.fit_transform(rank_matrix.to_numpy(dtype=float))
            coord_name = "UMAP"
        except Exception as exc:
            from sklearn.decomposition import PCA
            print(f"UMAP unavailable; falling back to PCA: {exc}")
            coords = PCA(n_components=2, random_state=42).fit_transform(rank_matrix.to_numpy(dtype=float))
            coord_name = "PCA"
        emb = pd.DataFrame({"SequenceIndex": rank_matrix.index.astype(str), f"{coord_name}1": coords[:, 0], f"{coord_name}2": coords[:, 1]})
        label_df = normalize_label_columns(active_labels)
        label_df = label_df[label_df["annotation"].isin(PATHOGENICITY_LABELS)].copy()
        emb = emb.merge(label_df[["SequenceIndex", "annotation", "stars"]], on="SequenceIndex", how="left", suffixes=("", "_label"))
        emb = normalize_label_columns(emb)
        fig, ax = plt.subplots(figsize=(7.2, 6.0))
        sns.scatterplot(data=emb, x=f"{coord_name}1", y=f"{coord_name}2", hue="annotation", style="annotation", alpha=0.82, s=38, ax=ax)
        ax.set_title(f"BRCA1 SAE ensemble component-rank {coord_name}")
        ax.legend(title="Clinical label", frameon=False)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_sae_ensemble_component_rank_{coord_name.lower()}_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)


## Output Checklist

Useful files after running the notebook:

- SAE metrics: `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/tables/MV_BRCA1_Findlay_2018_fixed_deltaembsae_layer_metrics.csv`
- LLR fitness tables: `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/tables/*_llr_fitness.csv`
- Benchmark metrics: `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/tables/MV_BRCA1_Findlay_2018_method_metrics.csv`
- Ensemble metrics: `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/tables/MV_BRCA1_Findlay_2018_sae_ensemble_metrics.csv`
- Notebook-specific figures: `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/figures/brca1_sae_downstream/`

## Reconstruction And Cross-Replicate Diagnostics

This builds one row per raw ESM-C layer and per DeltaEmbSAE layer with:

- `cross_replicate_consistency`: mean pairwise Pearson r between the per-replicate popDMS selection vectors (`InferenceResult.s`) each model produces. SAE rows read the pickle named in `inference_path`; raw ESM-C rows leave that column blank in the benchmark tables, so the cell resolves their cached inference pickle under `sequence_data/`.
- `mean_cosine_similarity`: mean per-sequence cosine similarity between the original and SAE-reconstructed embeddings (held-out split when available), from the SAE `viz_path`.
- `fraction_variance_explained`: `1 - SSE/SST` on the same split, i.e. the fraction of embedding variance the SAE reconstruction retains.

Raw ESM-C rows have no reconstruction (they are the reconstruction target), so their cosine/variance columns stay `NaN`.

The pickle reads are slow, so results are cached per `inference_path`. Set `REBUILD_MODEL_DIAGNOSTICS = True` to recompute from scratch.


In [ ]:
from scipy.stats import pearsonr, spearmanr

DIAGNOSTIC_REVIEW_STAR_CUTOFF = 0
REBUILD_MODEL_DIAGNOSTICS = False
MODEL_DIAGNOSTICS_CACHE = TABLE_DIR / f"{DATASET}_model_reconstruction_consistency_diagnostics.csv"
DIAGNOSTIC_COLUMNS = [
    "cross_replicate_consistency",
    "n_replicate_pairs",
    "mean_cosine_similarity",
    "fraction_variance_explained",
]

def load_pickle(path):
    with Path(path).open("rb") as handle:
        return pickle.load(handle)

def existing_path(value):
    if value is None or (isinstance(value, float) and np.isnan(value)) or pd.isna(value) or not str(value):
        return None
    path = Path(str(value))
    return path if path.is_file() else None

def resolve_inference_path(row):
    """Raw-embedding benchmark rows leave inference_path blank, but popDMS caches their inference pickle under sequence_data."""
    path = existing_path(row.get("inference_path"))
    if path is not None:
        return str(path)
    layer = safe_numeric(pd.Series([row.get("layer")])).iloc[0]
    model = str(row.get("model", ""))
    if not model or not np.isfinite(layer):
        return ""
    pattern = f"{DATASET}_{model.replace('/', '__')}_*_none_Layer_{int(layer)}_none_inference_results.pkl"
    matches = sorted((DATASET_ROOT / "sequence_data").glob(pattern))
    return str(matches[0]) if matches else ""

def mean_pairwise_pearson(matrix):
    arr = np.asarray(matrix, dtype=float)
    if arr.ndim != 2 or arr.shape[0] < 2:
        return np.nan, 0
    pair_corrs = []
    for i in range(arr.shape[0]):
        for j in range(i + 1, arr.shape[0]):
            finite = np.isfinite(arr[i]) & np.isfinite(arr[j])
            x, y = arr[i][finite], arr[j][finite]
            if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
                continue
            pair_corrs.append(float(pearsonr(x, y).statistic))
    if not pair_corrs:
        return np.nan, 0
    return float(np.mean(pair_corrs)), len(pair_corrs)

def inference_consistency(inference_path):
    """Mean pairwise Pearson r across the per-replicate popDMS selection vectors."""
    path = existing_path(inference_path)
    if path is None:
        return np.nan, 0
    s_replicates = getattr(load_pickle(path), "s", None)
    if s_replicates is None:
        return np.nan, 0
    return mean_pairwise_pearson(s_replicates)

def sae_reconstruction_quality(viz_path):
    """Cosine similarity and fraction of variance explained between original and SAE-reconstructed embeddings."""
    path = existing_path(viz_path)
    if path is None:
        return {"mean_cosine_similarity": np.nan, "fraction_variance_explained": np.nan}
    viz = load_pickle(path)
    x = np.asarray(viz["X_original"], dtype=float)
    x_recon = np.asarray(viz["X_reconstructed"], dtype=float)
    test_idx = np.asarray(viz.get("test_idx") or [], dtype=int)
    eval_idx = test_idx if len(test_idx) else np.arange(x.shape[0])
    x, x_recon = x[eval_idx], x_recon[eval_idx]
    norms = np.linalg.norm(x, axis=1) * np.linalg.norm(x_recon, axis=1)
    cosine = np.divide(np.einsum("ij,ij->i", x, x_recon), norms, out=np.full(len(x), np.nan), where=norms > 0)
    residual_ss = float(np.sum((x - x_recon) ** 2))
    total_ss = float(np.sum((x - x.mean(axis=0, keepdims=True)) ** 2))
    return {
        "mean_cosine_similarity": float(np.nanmean(cosine)) if np.isfinite(cosine).any() else np.nan,
        "fraction_variance_explained": 1.0 - residual_ss / total_ss if total_ss > 0 else np.nan,
    }

def build_model_diagnostics(auc_table, sae_layer_metrics, min_stars=DIAGNOSTIC_REVIEW_STAR_CUTOFF, rebuild=REBUILD_MODEL_DIAGNOSTICS):
    if auc_table.empty:
        return pd.DataFrame()
    rows = auc_table[
        auc_table["min_review_stars"].eq(min_stars)
        & auc_table["method_family"].isin(["Raw embeddings", "Raw SAE"])
    ].copy()
    if rows.empty:
        return pd.DataFrame()
    keep = [c for c in [
        "method_family", "benchmark", "model", "model_short", "layer", "model_label",
        "spearman_rho", "auc", "auc_discrimination", "n_benign", "n_pathogenic",
        "inference_path", "fitness_path",
    ] if c in rows.columns]
    rows = rows[keep].copy()
    rows["layer"] = safe_numeric(rows["layer"])
    rows["spearman_rho"] = safe_numeric(rows["spearman_rho"])
    rows["auc_discrimination"] = safe_numeric(rows["auc_discrimination"])
    rows["inference_path"] = rows.apply(resolve_inference_path, axis=1)

    # SAE viz pickles hold the reconstruction; join them onto the benchmark rows by inference_path.
    viz_lookup = {}
    if not sae_layer_metrics.empty and {"inference_path", "viz_path", "run_label"} <= set(sae_layer_metrics.columns):
        current = sae_layer_metrics[sae_layer_metrics["run_label"].astype(str).eq(RUN_LABEL)]
        viz_lookup = dict(zip(current["inference_path"].astype(str), current["viz_path"].astype(str)))
    rows["viz_path"] = rows["inference_path"].map(viz_lookup).fillna("")

    cache = {}
    cached = pd.DataFrame() if rebuild else read_csv(MODEL_DIAGNOSTICS_CACHE)
    if not cached.empty and "inference_path" in cached.columns:
        for _, cached_row in cached.iterrows():
            cache[str(cached_row["inference_path"])] = {col: cached_row.get(col, np.nan) for col in DIAGNOSTIC_COLUMNS}

    records = []
    n_computed = 0
    for _, row in rows.iterrows():
        key = row["inference_path"]
        values = cache.get(key)
        if values is None:
            rho, n_pairs = inference_consistency(key)
            values = {
                "cross_replicate_consistency": rho,
                "n_replicate_pairs": n_pairs,
                **sae_reconstruction_quality(row["viz_path"]),
            }
            cache[key] = values
            n_computed += 1
        records.append({**row.to_dict(), **values})

    diagnostics = pd.DataFrame(records)
    for col in DIAGNOSTIC_COLUMNS:
        diagnostics[col] = safe_numeric(diagnostics[col])
    diagnostics["method_plot_label"] = diagnostics["method_family"].map(method_label)
    diagnostics["is_best"] = mark_best_rows(diagnostics, best_auc_rows)
    print(f"Computed diagnostics for {n_computed} model rows; {len(diagnostics) - n_computed} came from the cache.")
    MODEL_DIAGNOSTICS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    diagnostics.to_csv(MODEL_DIAGNOSTICS_CACHE, index=False)
    return diagnostics

model_diagnostics = build_model_diagnostics(auc_by_labels, sae_metrics)
if model_diagnostics.empty:
    print("No raw-embedding/SAE rows available. Run the metrics/AUC recompute cells first.")
else:
    print(f"Diagnostic rows: {len(model_diagnostics)}")
    print(MODEL_DIAGNOSTICS_CACHE)
    display(
        model_diagnostics.groupby("method_plot_label")[
            ["auc_discrimination", "cross_replicate_consistency", "mean_cosine_similarity", "fraction_variance_explained"]
        ].agg(["count", "median", "max"])
    )
    display(
        model_diagnostics.sort_values("auc_discrimination", ascending=False)[
            ["method_plot_label", "model_short", "layer", "auc_discrimination", "spearman_rho",
             "cross_replicate_consistency", "mean_cosine_similarity", "fraction_variance_explained", "is_best"]
        ].head(10)
    )


## Plot: AUC Distribution For Raw ESM-C Layers Versus SAE Layers

Histogram of direction-normalized ClinVar AUC across every raw ESM-C layer and every DeltaEmbSAE layer, with each family's median and best layer marked, plus dotted guide lines for the non-model baselines.


In [ ]:
plt, sns = ensure_plotting()

if model_diagnostics.empty:
    print("No diagnostics to plot. Run the diagnostics cell first.")
else:
    dist_df = model_diagnostics[np.isfinite(model_diagnostics["auc_discrimination"])].copy()
    if dist_df.empty:
        print("No finite AUC values to plot.")
    else:
        baselines = auc_by_labels[
            auc_by_labels["min_review_stars"].eq(DIAGNOSTIC_REVIEW_STAR_CUTOFF)
            & auc_by_labels["method_family"].isin(["popDMS baseline", "LLR baseline", "Ensemble SAE model"])
        ].copy()
        if not baselines.empty:
            baselines["auc_discrimination"] = safe_numeric(baselines["auc_discrimination"])
            baselines = (
                baselines[np.isfinite(baselines["auc_discrimination"])]
                .sort_values("auc_discrimination", ascending=False)
                .drop_duplicates("method_family")
            )

        fig, ax = plt.subplots(figsize=(10.6, 6.0))
        bins = np.histogram_bin_edges(dist_df["auc_discrimination"].to_numpy(dtype=float), bins=24)
        for family in ["Raw embeddings", "Raw SAE"]:
            values = dist_df.loc[dist_df["method_family"].eq(family), "auc_discrimination"].to_numpy(dtype=float)
            if not len(values):
                continue
            color = method_color(family)
            ax.hist(
                values,
                bins=bins,
                color=color,
                alpha=0.45,
                edgecolor="white",
                linewidth=0.5,
                label=f"{method_label(family)} (n={len(values)})",
                zorder=2,
            )
            ax.axvline(
                np.median(values), color=color, linestyle="--", linewidth=1.5, alpha=0.9, zorder=4,
                label=f"{method_label(family)} median AUC*={np.median(values):.3f}",
            )
            # Best layer per family, same selection as the other AUC plots.
            for _, row in dist_df[dist_df["method_family"].eq(family) & dist_df["is_best"]].iterrows():
                layer_text = "" if pd.isna(row.get("layer")) else f" L{int(row['layer'])}"
                ax.axvline(
                    row["auc_discrimination"], color=color, linewidth=2.2, alpha=0.98, zorder=5,
                    label=f"Best {method_label(family)}: {row.get('model_short', '')}{layer_text} (AUC*={row['auc_discrimination']:.3f})",
                )

        for _, row in baselines.iterrows():
            family = row["method_family"]
            ax.axvline(row["auc_discrimination"], color=method_color(family), linestyle=":", linewidth=1.3, alpha=0.75, zorder=1)
            ax.text(
                row["auc_discrimination"],
                0.985,
                f"{method_label(family)} {row['auc_discrimination']:.3f} ",
                transform=ax.get_xaxis_transform(),
                rotation=90,
                va="top",
                ha="right",
                fontsize=8,
                color=method_color(family),
            )

        ax.axvline(0.5, color="0.72", linewidth=1, linestyle="--", zorder=1)
        ax.set_xlabel("Direction-normalized ClinVar AUC")
        ax.set_ylabel("Number of layers")
        cutoff_label = "all binary labels" if DIAGNOSTIC_REVIEW_STAR_CUTOFF == 0 else f">={DIAGNOSTIC_REVIEW_STAR_CUTOFF} review stars"
        ax.set_title(f"BRCA1 AUC distribution across layers ({CLINICAL_LABEL_MODE}, {cutoff_label})")
        ax.legend(title="Model family", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_auc_distribution_raw_vs_sae_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)
        display(
            dist_df.groupby("method_plot_label")["auc_discrimination"]
            .agg(["count", "min", "median", "mean", "max"])
        )


## Plot: AUC Versus SAE Reconstruction Quality

Direction-normalized ClinVar AUC against two reconstruction metrics for the DeltaEmbSAE layers: mean per-sequence cosine similarity and fraction of variance explained. The best SAE layer by AUC is opaque; the remaining layers are transparent, matching the Spearman-versus-AUC plot. Each panel reports the Spearman correlation between the reconstruction metric and AUC across layers.


In [ ]:
plt, sns = ensure_plotting()

if model_diagnostics.empty:
    print("No diagnostics to plot. Run the diagnostics cell first.")
else:
    sae_df = model_diagnostics[model_diagnostics["method_family"].eq("Raw SAE")].copy()
    sae_df = sae_df[np.isfinite(sae_df["auc_discrimination"])].copy()
    if sae_df.empty:
        print("No SAE rows with finite AUC.")
    else:
        color = method_color("Raw SAE")
        marker = method_marker("Raw SAE")
        label_base = method_label("Raw SAE")
        metric_specs = [
            ("mean_cosine_similarity", "Mean per-sequence cosine similarity (original vs reconstructed)"),
            ("fraction_variance_explained", "Fraction of variance explained by reconstruction"),
        ]

        fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.8), sharey=True)
        for ax, (metric, xlabel) in zip(axes, metric_specs):
            panel = sae_df[np.isfinite(sae_df[metric])].copy()
            if panel.empty:
                ax.text(0.5, 0.5, f"No finite {metric} values", ha="center", va="center", transform=ax.transAxes)
                ax.set_xlabel(xlabel)
                continue

            nonbest = panel[~panel["is_best"]]
            if not nonbest.empty:
                ax.scatter(
                    nonbest[metric], nonbest["auc_discrimination"],
                    s=46, marker=marker, color=color, alpha=0.18, edgecolor="none",
                    label=f"{label_base} all layers", zorder=2,
                )
            best = panel[panel["is_best"]]
            if not best.empty:
                ax.scatter(
                    best[metric], best["auc_discrimination"],
                    s=180, marker=marker, color=color, alpha=0.98, edgecolor="black", linewidth=0.9,
                    label=f"Best {label_base}", zorder=6,
                )
                for _, row in best.iterrows():
                    layer_text = "" if pd.isna(row.get("layer")) else f" L{int(row['layer'])}"
                    ax.annotate(
                        f"best {label_base}\n{row.get('model_short', '')}{layer_text}",
                        (row[metric], row["auc_discrimination"]),
                        xytext=(7, 6), textcoords="offset points", fontsize=8, color="0.15",
                    )

            rho = spearmanr(panel[metric], panel["auc_discrimination"]).statistic if len(panel) >= 3 else np.nan
            ax.axhline(0.5, color="0.72", linewidth=1, linestyle="--", zorder=1)
            ax.set_xlabel(xlabel)
            ax.set_title(f"Spearman(metric, AUC*) = {rho:.3f} over {len(panel)} SAE layers")
            ax.legend(frameon=False, loc="lower right")

        axes[0].set_ylabel("Direction-normalized ClinVar AUC")
        axes[0].set_ylim(0.45, 1.02)
        cutoff_label = "all binary labels" if DIAGNOSTIC_REVIEW_STAR_CUTOFF == 0 else f">={DIAGNOSTIC_REVIEW_STAR_CUTOFF} review stars"
        fig.suptitle(f"BRCA1 downstream AUC vs SAE reconstruction quality ({CLINICAL_LABEL_MODE}, {cutoff_label})", y=1.02)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_auc_vs_sae_reconstruction_metrics_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)


## Plot: AUC Versus Cross-Replicate Consistency

Cross-replicate consistency (mean pairwise Pearson r between the per-replicate popDMS selection vectors each model produces) against direction-normalized ClinVar AUC, for every raw ESM-C layer and every DeltaEmbSAE layer. Best layers per family are opaque, the rest transparent. The popDMS baseline is drawn as its own point since it reports both quantities, and the LLR baseline appears as a dotted AUC guide line.


In [ ]:
plt, sns = ensure_plotting()

if model_diagnostics.empty:
    print("No diagnostics to plot. Run the diagnostics cell first.")
else:
    consistency_df = model_diagnostics[
        np.isfinite(model_diagnostics["cross_replicate_consistency"])
        & np.isfinite(model_diagnostics["auc_discrimination"])
    ].copy()
    if consistency_df.empty:
        print("No models with a finite cross-replicate consistency. Check that the inference pickles exist.")
    else:
        fig, ax = plt.subplots(figsize=(9.8, 6.5))

        for family in ["Raw embeddings", "Raw SAE"]:
            group = consistency_df[consistency_df["method_family"].eq(family)]
            if group.empty:
                continue
            color = method_color(family)
            marker = method_marker(family)
            label_base = method_label(family)
            nonbest = group[~group["is_best"]]
            if not nonbest.empty:
                ax.scatter(
                    nonbest["cross_replicate_consistency"], nonbest["auc_discrimination"],
                    s=46, marker=marker, color=color, alpha=0.18, edgecolor="none",
                    label=f"{label_base} all layers", zorder=2,
                )
            # Best points of the two families sit almost on top of each other, so name them in the legend.
            for _, row in group[group["is_best"]].iterrows():
                layer_text = "" if pd.isna(row.get("layer")) else f" L{int(row['layer'])}"
                ax.scatter(
                    row["cross_replicate_consistency"], row["auc_discrimination"],
                    s=180, marker=marker, color=color, alpha=0.98, edgecolor="black", linewidth=0.9,
                    label=f"Best {label_base}: {row.get('model_short', '')}{layer_text} (r={row['cross_replicate_consistency']:.3f}, AUC*={row['auc_discrimination']:.3f})",
                    zorder=6,
                )

        cutoff_rows = auc_by_labels[auc_by_labels["min_review_stars"].eq(DIAGNOSTIC_REVIEW_STAR_CUTOFF)].copy()
        cutoff_rows["auc_discrimination"] = safe_numeric(cutoff_rows["auc_discrimination"])

        popdms = cutoff_rows[cutoff_rows["method_family"].eq("popDMS baseline")].copy()
        if not popdms.empty and "mean_pairwise_pearson_r" in popdms.columns:
            popdms["mean_pairwise_pearson_r"] = safe_numeric(popdms["mean_pairwise_pearson_r"])
            popdms = popdms[np.isfinite(popdms["mean_pairwise_pearson_r"]) & np.isfinite(popdms["auc_discrimination"])]
            for _, row in popdms.iterrows():
                ax.scatter(
                    row["mean_pairwise_pearson_r"], row["auc_discrimination"],
                    s=145, marker=method_marker("popDMS baseline"), color=method_color("popDMS baseline"),
                    alpha=0.94, edgecolor="black", linewidth=0.8, zorder=5,
                    label=f"{method_label('popDMS baseline')} (r={row['mean_pairwise_pearson_r']:.3f}, AUC*={row['auc_discrimination']:.3f})",
                )

        guide_df = cutoff_rows[cutoff_rows["method_family"].isin(["LLR baseline", "Ensemble SAE model"])].copy()
        guide_df = (
            guide_df[np.isfinite(guide_df["auc_discrimination"])]
            .sort_values("auc_discrimination", ascending=False)
            .drop_duplicates("method_family")
        )
        for _, row in guide_df.iterrows():
            family = row["method_family"]
            y = row["auc_discrimination"]
            ax.axhline(y, color=method_color(family), linestyle=":", linewidth=1.15, alpha=0.65, zorder=0)
            ax.text(
                0.01, y, f"{method_label(family)} AUC*={y:.3f}",
                transform=ax.get_yaxis_transform(), va="bottom", ha="left", fontsize=8, color=method_color(family),
            )

        rho = (
            spearmanr(consistency_df["cross_replicate_consistency"], consistency_df["auc_discrimination"]).statistic
            if len(consistency_df) >= 3 else np.nan
        )
        ax.axhline(0.5, color="0.72", linewidth=1, linestyle="--", zorder=1)
        ax.set_ylim(0.45, 1.02)
        ax.set_xlabel("Cross-replicate consistency (mean pairwise Pearson r of per-replicate selection)")
        ax.set_ylabel("Direction-normalized ClinVar AUC")
        cutoff_label = "all binary labels" if DIAGNOSTIC_REVIEW_STAR_CUTOFF == 0 else f">={DIAGNOSTIC_REVIEW_STAR_CUTOFF} review stars"
        ax.set_title(
            f"BRCA1 cross-replicate consistency vs AUC ({CLINICAL_LABEL_MODE}, {cutoff_label})\n"
            f"Spearman(consistency, AUC*) = {rho:.3f} over {len(consistency_df)} layers"
        )

        handles, labels = ax.get_legend_handles_labels()
        seen = set()
        dedup_handles, dedup_labels = [], []
        for handle, label in zip(handles, labels):
            if label in seen or label == "":
                continue
            seen.add(label)
            dedup_handles.append(handle)
            dedup_labels.append(label)
        ax.legend(dedup_handles, dedup_labels, title="Point set", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)

        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_auc_vs_cross_replicate_consistency_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)
        display(
            consistency_df.groupby("method_plot_label")[["cross_replicate_consistency", "auc_discrimination"]]
            .agg(["count", "median", "max"])
        )


## Plot: SAE Reconstruction Quality By Layer And ESM-C Model

Layer profile of the two reconstruction metrics, one line per ESM-C model. This is the control for the AUC-versus-reconstruction plots: downstream AUC tracks layer depth very strongly (Spearman ~0.94), so any pooled correlation between AUC and reconstruction quality is confounded by depth, and the two models have different layer counts. Best-by-AUC layers are marked.


In [ ]:
plt, sns = ensure_plotting()

MODEL_COLORS = {"ESMC-300M": "#4C78A8", "ESMC-600M": "#E45756"}

def model_color(value):
    return MODEL_COLORS.get(str(value), "0.4")

if model_diagnostics.empty:
    print("No diagnostics to plot. Run the diagnostics cell first.")
else:
    recon_df = model_diagnostics[model_diagnostics["method_family"].eq("Raw SAE")].copy()
    recon_df["layer"] = safe_numeric(recon_df["layer"])
    recon_df = recon_df[np.isfinite(recon_df["layer"])].copy()
    if recon_df.empty:
        print("No SAE rows with a layer index.")
    else:
        metric_specs = [
            ("mean_cosine_similarity", "Mean per-sequence cosine similarity", None),
            ("fraction_variance_explained", "Fraction of variance explained", 0.0),
        ]
        fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.6), sharex=True)
        for ax, (metric, ylabel, refline) in zip(axes, metric_specs):
            for model_short, group in recon_df.groupby("model_short", sort=True):
                group = group[np.isfinite(group[metric])].sort_values("layer")
                if group.empty:
                    continue
                color = model_color(model_short)
                ax.plot(group["layer"], group[metric], color=color, alpha=0.85, linewidth=1.6, marker="^", markersize=5, label=f"{model_short} (n={len(group)} layers)")
                best = group[group["is_best"]]
                if not best.empty:
                    ax.scatter(
                        best["layer"], best[metric],
                        s=170, marker="^", color=color, alpha=0.98, edgecolor="black", linewidth=0.9, zorder=6,
                        label=f"Best-AUC layer ({model_short} L{int(best.iloc[0]['layer'])})",
                    )
            if refline is not None:
                ax.axhline(refline, color="0.65", linewidth=1, linestyle="--", zorder=1)
            ax.set_xlabel("ESM-C layer index")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{ylabel} by layer")
            ax.legend(frameon=False, fontsize=9, loc="lower left")

        fig.suptitle(f"BRCA1 DeltaEmbSAE reconstruction quality by layer ({RUN_LABEL})", y=1.02)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_sae_reconstruction_by_layer.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)
        display(
            recon_df.groupby("model_short")[["mean_cosine_similarity", "fraction_variance_explained", "auc_discrimination"]]
            .agg(["count", "min", "median", "max"])
        )


## Voting Ensembles From Model-Specific Pathogenicity Calls

Each raw ESM-C or DeltaEmbSAE layer orients `-fitness` so its all-binary ClinVar AUC is at least 0.5, then chooses a binary cutoff by maximum Youden's J (sensitivity minus false-positive rate). Those fixed 0/1 calls are averaged for the top 5, 9, 13, 17, 21, and 25 models by Spearman rho. Three searches are run: SAE only, raw embeddings only, and both families together.

Five targeted combined-family variants are also evaluated: top 3; top 3 plus both ESM-C 300M/600M LLR votes; top 3 plus ESM-C 600M LLR and popDMS; top 5 plus ESM-C 600M LLR and popDMS; and top 4 plus both ESM-C 300M/600M LLR votes and popDMS. Every added baseline gets its own AUC orientation and Youden cutoff. The all-binary labels fit each model's cutoff once; review-star panels only re-evaluate the fixed calls.


In [ ]:
from scipy.stats import spearmanr

VOTING_TOP_NS = [5, 9, 13, 17, 21, 25]
VOTING_POOLS = {
    "SAE only": ["Raw SAE"],
    "Raw embeddings only": ["Raw embeddings"],
    "Raw + SAE": ["Raw embeddings", "Raw SAE"],
}
TARGETED_VOTING_SPECS = [
    ("Both top 3", 3, []),
    ("Both top 3 + 300M LLR + 600M LLR", 3, ["LLR ESMC-300M", "LLR ESMC-600M"]),
    ("Both top 3 + 600M LLR + popDMS", 3, ["LLR ESMC-600M", "popDMS"]),
    ("Both top 5 + 600M LLR + popDMS", 5, ["LLR ESMC-600M", "popDMS"]),
    ("Both top 4 + 300M LLR + 600M LLR + popDMS", 4, ["LLR ESMC-300M", "LLR ESMC-600M", "popDMS"]),
]
VOTING_TABLE_DIR = TABLE_DIR / "voting_ensembles"
WRITE_VOTING_TABLES = True


def score_auc(scores, labels):
    scores = safe_numeric(pd.Series(scores)).to_numpy(dtype=float)
    labels = pd.Series(labels).astype(bool).to_numpy()
    keep = np.isfinite(scores)
    scores, labels = scores[keep], labels[keep]
    n_pos, n_neg = int(labels.sum()), int((~labels).sum())
    if not n_pos or not n_neg:
        return np.nan
    ranks = pd.Series(scores).rank(method="average").to_numpy()
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def youden_cutoff(scores, labels):
    table = pd.DataFrame({"score": safe_numeric(pd.Series(scores)), "positive": pd.Series(labels).astype(bool)})
    table = table[np.isfinite(table["score"])]
    n_pos, n_neg = int(table["positive"].sum()), int((~table["positive"]).sum())
    if not n_pos or not n_neg:
        return np.nan
    grouped = table.groupby("score")["positive"].agg(positive="sum", total="size").sort_index(ascending=False)
    youden = grouped["positive"].cumsum() / n_pos - (grouped["total"] - grouped["positive"]).cumsum() / n_neg
    return float(youden.idxmax())


def classify_for_voting(row):
    joined = fitness_with_active_labels(row["fitness_path"], min_stars=0)
    joined["raw_pathogenicity_score"] = -safe_numeric(joined.get("fitness", np.nan))
    train = joined[joined["annotation"].isin(PATHOGENICITY_LABELS)].dropna(subset=["raw_pathogenicity_score"])
    labels = train["annotation"].eq("pathogenic")
    raw_auc = score_auc(train["raw_pathogenicity_score"], labels)
    if not np.isfinite(raw_auc):
        return pd.DataFrame(), None
    direction = 1.0 if raw_auc >= 0.5 else -1.0
    cutoff = youden_cutoff(direction * train["raw_pathogenicity_score"], labels)
    if not np.isfinite(cutoff):
        return pd.DataFrame(), None
    calls = joined[["SequenceIndex"]].copy()
    calls["SequenceIndex"] = calls["SequenceIndex"].astype(str)
    calls["pathogenicity_score"] = direction * joined["raw_pathogenicity_score"]
    calls = calls[np.isfinite(calls["pathogenicity_score"])].copy()
    calls["pathogenic_vote"] = calls["pathogenicity_score"].ge(cutoff).astype(int)
    for column in ["voting_model_id", "method_family", "model_label", "model_short", "layer", "spearman_rho", "fitness_path"]:
        calls[column] = row.get(column, np.nan)
    calls["model_auc"] = raw_auc
    calls["auc_direction"] = direction
    calls["youden_cutoff"] = cutoff
    metadata = {column: row.get(column, np.nan) for column in ["voting_model_id", "method_family", "model_label", "model_short", "layer", "spearman_rho", "fitness_path"]}
    metadata.update(model_auc=raw_auc, auc_discrimination=max(raw_auc, 1 - raw_auc), auc_direction=direction, youden_cutoff=cutoff, n_classified_variants=len(calls))
    return calls, metadata


def voting_stats(predictions, target_scores):
    scored = predictions[["SequenceIndex", "vote_fraction"]].copy()
    scored["fitness"] = -scored["vote_fraction"]
    merged = scored.merge(target_scores, on="SequenceIndex", how="inner")
    keep = np.isfinite(merged["fitness"]) & np.isfinite(merged["score"])
    rho = spearmanr(merged.loc[keep, "fitness"], merged.loc[keep, "score"]).statistic if keep.sum() >= 3 else np.nan
    labels = normalize_label_columns(active_labels.copy())
    labels["SequenceIndex"] = labels["SequenceIndex"].astype(str)
    rows = []
    for min_stars in [0, 1, 2, 3]:
        cutoff_labels = labels[labels["stars"].ge(min_stars) & labels["annotation"].isin(PATHOGENICITY_LABELS)][["SequenceIndex", "annotation", "stars"]]
        evaluated = scored.merge(cutoff_labels, on="SequenceIndex", how="left")
        rows.append({"min_review_stars": min_stars, "spearman_rho": rho, **auc_from_labeled_fitness(evaluated)})
    return rows


voting_candidates = metrics[metrics["method_family"].isin(["Raw embeddings", "Raw SAE"])].copy()
voting_candidates["spearman_rho"] = safe_numeric(voting_candidates["spearman_rho"])
voting_candidates = voting_candidates[np.isfinite(voting_candidates["spearman_rho"])]
voting_candidates = voting_candidates[voting_candidates["fitness_path"].map(lambda path: Path(str(path)).is_file())]
voting_candidates = voting_candidates.sort_values("spearman_rho", ascending=False).drop_duplicates("fitness_path")
voting_candidates["voting_model_id"] = voting_candidates["method_family"].astype(str) + " | " + voting_candidates["model_label"].astype(str) + " | " + voting_candidates["fitness_path"].astype(str)

target_scores = load_dataset_state()["scores_dataframe"][["SequenceIndex", "score"]].copy()
target_scores["SequenceIndex"] = target_scores["SequenceIndex"].astype(str)
target_scores["score"] = safe_numeric(target_scores["score"])


def rho_for_fitness_path(fitness_path):
    fitness = read_csv(fitness_path)
    if fitness.empty or not {"SequenceIndex", "fitness"}.issubset(fitness.columns):
        return np.nan
    fitness = fitness[["SequenceIndex", "fitness"]].copy()
    fitness["SequenceIndex"] = fitness["SequenceIndex"].astype(str)
    merged = fitness.merge(target_scores, on="SequenceIndex", how="inner")
    keep = np.isfinite(safe_numeric(merged["fitness"])) & np.isfinite(merged["score"])
    return float(spearmanr(safe_numeric(merged.loc[keep, "fitness"]), merged.loc[keep, "score"]).statistic) if keep.sum() >= 3 else np.nan


external_rows = []
popdms = metrics[metrics["method_family"].eq("popDMS baseline")].sort_values("spearman_rho", ascending=False).head(1)
if not popdms.empty:
    row = popdms.iloc[0].to_dict()
    row.update(voting_model_id="external | popDMS", voting_component_label="popDMS")
    external_rows.append(row)
for llr_path in sorted(TABLE_DIR.glob(f"{DATASET}_*_llr_fitness.csv")):
    model_short = "ESMC-600M" if "600M" in llr_path.name else "ESMC-300M"
    label = f"LLR {model_short}"
    external_rows.append({
        "voting_model_id": f"external | {label}", "voting_component_label": label,
        "method_family": "LLR baseline", "model_label": label, "model_short": model_short,
        "layer": np.nan, "spearman_rho": rho_for_fitness_path(llr_path), "fitness_path": str(llr_path),
    })
external_candidates = pd.DataFrame(external_rows)

call_frames, threshold_rows = [], []
for _, candidate in pd.concat([voting_candidates, external_candidates], ignore_index=True, sort=False).iterrows():
    calls, threshold = classify_for_voting(candidate)
    if not calls.empty:
        call_frames.append(calls)
        threshold_rows.append(threshold)
voting_model_calls = pd.concat(call_frames, ignore_index=True) if call_frames else pd.DataFrame()
voting_model_thresholds = pd.DataFrame(threshold_rows)

prediction_frames, metric_rows, selection_rows = [], [], []
classifiable_ids = set(voting_model_thresholds.get("voting_model_id", []))
for pool, families in VOTING_POOLS.items():
    ranked = voting_candidates[voting_candidates["method_family"].isin(families) & voting_candidates["voting_model_id"].isin(classifiable_ids)]
    for top_n in VOTING_TOP_NS:
        selected = ranked.head(top_n)
        if len(selected) < top_n:
            print(f"Skipping {pool} top {top_n}: only {len(selected)} classifiable models")
            continue
        for rank, (_, model) in enumerate(selected.iterrows(), 1):
            selection_rows.append({"ensemble_pool": pool, "top_n": top_n, "rank": rank, **model.to_dict()})
        votes = voting_model_calls[voting_model_calls["voting_model_id"].isin(selected["voting_model_id"])]
        predictions = votes.groupby("SequenceIndex", as_index=False)["pathogenic_vote"].agg(vote_fraction="mean", n_votes="size")
        predictions = predictions[predictions["n_votes"].eq(top_n)].copy()
        predictions["ensemble_pool"], predictions["top_n"] = pool, top_n
        predictions["ensemble_label"] = f"{pool} top {top_n}"
        predictions["n_constituent_models"] = top_n
        prediction_frames.append(predictions)
        for stats in voting_stats(predictions, target_scores):
            metric_rows.append({"method_family": "Voting ensemble", "ensemble_pool": pool, "ensemble_label": f"{pool} top {top_n}", "top_n": top_n, "n_constituent_models": top_n, **stats})

combined_ranked = voting_candidates[voting_candidates["method_family"].isin(VOTING_POOLS["Raw + SAE"]) & voting_candidates["voting_model_id"].isin(classifiable_ids)]
external_by_label = {row["voting_component_label"]: row for _, row in external_candidates[external_candidates["voting_model_id"].isin(classifiable_ids)].iterrows()}
for ensemble_label, base_top_n, extra_labels in TARGETED_VOTING_SPECS:
    base = combined_ranked.head(base_top_n)
    missing = [label for label in extra_labels if label not in external_by_label]
    if len(base) < base_top_n or missing:
        print(f"Skipping {ensemble_label}: missing {missing or 'combined candidates'}")
        continue
    selected_rows = [row for _, row in base.iterrows()] + [external_by_label[label] for label in extra_labels]
    selected_ids = [row["voting_model_id"] for row in selected_rows]
    total_votes = len(selected_ids)
    for rank, model in enumerate(selected_rows, 1):
        selection_rows.append({"ensemble_pool": "Targeted combinations", "ensemble_label": ensemble_label, "top_n": base_top_n, "n_constituent_models": total_votes, "rank": rank, **model.to_dict()})
    votes = voting_model_calls[voting_model_calls["voting_model_id"].isin(selected_ids)]
    predictions = votes.groupby("SequenceIndex", as_index=False)["pathogenic_vote"].agg(vote_fraction="mean", n_votes="size")
    predictions = predictions[predictions["n_votes"].eq(total_votes)].copy()
    predictions["ensemble_pool"] = "Targeted combinations"
    predictions["ensemble_label"] = ensemble_label
    predictions["top_n"] = base_top_n
    predictions["n_constituent_models"] = total_votes
    prediction_frames.append(predictions)
    for stats in voting_stats(predictions, target_scores):
        metric_rows.append({"method_family": "Voting ensemble", "ensemble_pool": "Targeted combinations", "ensemble_label": ensemble_label, "top_n": base_top_n, "n_constituent_models": total_votes, **stats})

voting_ensemble_predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
voting_ensemble_metrics = pd.DataFrame(metric_rows)
voting_ensemble_selections = pd.DataFrame(selection_rows)

if WRITE_VOTING_TABLES and not voting_model_calls.empty:
    VOTING_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    for name, table in {
        "model_pathogenicity_calls": voting_model_calls,
        "model_pathogenicity_thresholds": voting_model_thresholds,
        "voting_ensemble_selections": voting_ensemble_selections,
        "voting_ensemble_predictions": voting_ensemble_predictions,
        "voting_ensemble_metrics": voting_ensemble_metrics,
    }.items():
        table.to_csv(VOTING_TABLE_DIR / f"{DATASET}_{name}.csv", index=False)
    print(f"Voting tables: {VOTING_TABLE_DIR}")
     
print(f"Classified {len(voting_model_thresholds)} models; created {len(voting_ensemble_metrics) // 4} voting ensembles")
display(voting_model_thresholds.sort_values(["method_family", "spearman_rho"], ascending=[True, False]).head(12))
display(voting_ensemble_metrics)


## Plot: Spearman Versus AUC By ClinVar Review-Star Cutoff

The Spearman-versus-AUC scatter repeated at four ClinVar review-star cutoffs (>=0, >=1, >=2, >=3 stars). Everything is recomputed per panel: the AUC of every layer, the best raw ESM-C and best DeltaEmbSAE layer, the DMS-score / LLR guide lines, and the SAE-only, raw-only, and combined voting ensembles. Spearman rho is unchanged across panels (it is scored against the DMS functional score, not the clinical labels), so the points move only along the y-axis as the label set gets stricter and smaller. Voting-ensemble points are connected within each search pool and labeled by N.

The LLR baselines are drawn as points as well as AUC guide lines. Their Spearman rho is computed here from the `*_llr_fitness.csv` tables against the DMS functional score, using the same procedure as the pipeline's collected rows, so it is comparable to every other point.


In [ ]:
from scipy.stats import spearmanr

plt, sns = ensure_plotting()

GRID_STAR_CUTOFFS = [0, 1, 2, 3]

def spearman_target_scores():
    """The DMS functional score every method's Spearman rho is measured against."""
    state = load_dataset_state()
    scores = state["scores_dataframe"][["SequenceIndex", "score"]].copy()
    scores["SequenceIndex"] = scores["SequenceIndex"].astype(str)
    scores["score"] = safe_numeric(scores["score"])
    return scores.dropna(subset=["score"])

def spearman_vs_functional_score(fitness_path, scores):
    """Mirrors the pipeline's spearman_for_fitness so LLR rho is comparable to the collected rows."""
    fitness = read_csv(fitness_path)
    if fitness.empty or "SequenceIndex" not in fitness.columns or "fitness" not in fitness.columns:
        return np.nan
    fitness = fitness[["SequenceIndex", "fitness"]].copy()
    fitness["SequenceIndex"] = fitness["SequenceIndex"].astype(str)
    merged = fitness.merge(scores, on="SequenceIndex", how="inner")
    x = safe_numeric(merged["fitness"]).to_numpy(dtype=float)
    y = safe_numeric(merged["score"]).to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if finite.sum() < 3:
        return np.nan
    return float(spearmanr(x[finite], y[finite]).statistic)

def llr_rows_for_cutoff(panel_df, min_stars, scores):
    """LLR Spearman rho and AUC at a star cutoff, from benchmark rows if collected, else from the LLR fitness tables."""
    rows = panel_df[panel_df["method_family"].eq("LLR baseline")].copy()
    if not rows.empty:
        return [
            {
                "label": str(row.get("model_label") or row.get("model_short") or method_label("LLR baseline")),
                "spearman_rho": float(row["spearman_rho"]),
                "auc_discrimination": float(row["auc_discrimination"]),
            }
            for _, row in rows.iterrows()
        ]
    out = []
    for path in sorted(TABLE_DIR.glob(f"{DATASET}_*_llr_fitness.csv")):
        stats = auc_from_labeled_fitness(fitness_with_active_labels(path, min_stars=min_stars))
        if not np.isfinite(stats["auc_discrimination"]):
            continue
        model_tag = path.stem.replace(f"{DATASET}_", "").replace("_llr_fitness", "").split("__")[-1]
        out.append({
            "label": f"{method_label('LLR baseline')} {model_tag}",
            "spearman_rho": spearman_vs_functional_score(path, scores),  # label-independent, same at every cutoff
            "auc_discrimination": stats["auc_discrimination"],
        })
    return out

def draw_spearman_auc_panel(ax, min_stars, scores):
    panel_df = auc_by_labels[auc_by_labels["min_review_stars"].eq(min_stars)].copy()
    panel_df["spearman_rho"] = safe_numeric(panel_df.get("spearman_rho", np.nan))
    panel_df["auc_discrimination"] = safe_numeric(panel_df["auc_discrimination"])
    panel_df["layer"] = safe_numeric(panel_df.get("layer", np.nan))

    # Best-by-AUC rows are re-selected at each cutoff, so the highlighted layer can change between panels.
    panel_best = best_rows_by_family(metrics, auc_by_labels, min_stars=min_stars)
    panel_df["is_best"] = mark_best_rows(panel_df, panel_best)

    scatter_df = panel_df[panel_df["method_family"].isin(METHOD_ORDER)].copy()
    scatter_df = scatter_df[np.isfinite(scatter_df["spearman_rho"]) & np.isfinite(scatter_df["auc_discrimination"])].copy()
    if scatter_df.empty:
        ax.text(0.5, 0.5, f"No rows at >={min_stars} stars", ha="center", va="center", transform=ax.transAxes)
        return 0, 0

    layer_df = scatter_df[scatter_df["method_family"].isin(["Raw embeddings", "Raw SAE"])]
    baseline_df = scatter_df[scatter_df["method_family"].isin(["Enrichment ratio baseline", "popDMS baseline"])]
    ensemble_df = scatter_df[scatter_df["method_family"].eq("Ensemble SAE model")]

    for family, group in layer_df.groupby("method_family", sort=False):
        color = method_color(family)
        marker = method_marker(family)
        label_base = method_label(family)
        nonbest = group[~group["is_best"]]
        if not nonbest.empty:
            ax.scatter(
                nonbest["spearman_rho"], nonbest["auc_discrimination"],
                s=30, marker=marker, color=color, alpha=0.18, edgecolor="none",
                label=f"{label_base} all layers", zorder=2,
            )
        for _, row in group[group["is_best"]].iterrows():
            layer_text = "" if pd.isna(row.get("layer")) else f" L{int(row['layer'])}"
            model_text = str(row.get("model_short", "")).replace("nan", "")
            ax.scatter(
                row["spearman_rho"], row["auc_discrimination"],
                s=130, marker=marker, color=color, alpha=0.98, edgecolor="black", linewidth=0.9,
                label=f"Best {label_base}: {model_text}{layer_text} ({row['auc_discrimination']:.3f})", zorder=6,
            )

    for _, row in baseline_df.iterrows():
        family = row["method_family"]
        ax.scatter(
            row["spearman_rho"], row["auc_discrimination"],
            s=105, marker=method_marker(family), color=method_color(family),
            alpha=0.94, edgecolor="black", linewidth=0.8,
            label=f"{method_label(family)} ({row['auc_discrimination']:.3f})", zorder=5,
        )

    for _, row in ensemble_df.iterrows():
        family = row["method_family"]
        ax.scatter(
            row["spearman_rho"], row["auc_discrimination"],
            s=160, marker=method_marker(family), color=method_color(family),
            alpha=0.98, edgecolor="black", linewidth=1.0,
            label=f"{method_label(family)} gamma=1 ({row['auc_discrimination']:.3f})", zorder=8,
        )

    voting_styles = {
        "SAE only": ("#2CA02C", "^"),
        "Raw embeddings only": ("#1F77B4", "o"),
        "Raw + SAE": ("#9467BD", "*"),
        "Targeted combinations": ("#D62728", "D"),
    }
    voting_panel = voting_ensemble_metrics[voting_ensemble_metrics["min_review_stars"].eq(min_stars)].copy()
    voting_panel["spearman_rho"] = safe_numeric(voting_panel["spearman_rho"])
    voting_panel["auc_discrimination"] = safe_numeric(voting_panel["auc_discrimination"])
    voting_panel = voting_panel[np.isfinite(voting_panel["spearman_rho"]) & np.isfinite(voting_panel["auc_discrimination"])]
    for pool, group in voting_panel.groupby("ensemble_pool", sort=False):
        group = group.sort_values("top_n")
        color, marker = voting_styles[pool]
        if pool == "Targeted combinations":
            ax.scatter(group["spearman_rho"], group["auc_discrimination"], s=95, marker=marker, color=color, alpha=0.98, edgecolor="black", linewidth=0.7, label="Targeted voting combinations", zorder=10)
            for _, vote_row in group.iterrows():
                ax.annotate(vote_row["ensemble_label"].replace("Both ", "").replace(" + ", "+"), (vote_row["spearman_rho"], vote_row["auc_discrimination"]), xytext=(4, 4), textcoords="offset points", fontsize=5.5, color=color, zorder=11)
            continue
        ax.plot(group["spearman_rho"], group["auc_discrimination"], color=color, linewidth=1.2, alpha=0.7, zorder=7)
        ax.scatter(
            group["spearman_rho"], group["auc_discrimination"], s=85, marker=marker,
            color=color, alpha=0.95, edgecolor="black", linewidth=0.6,
            label=f"Voting: {pool} (N=5..25)", zorder=9,
        )
        for _, vote_row in group.iterrows():
            ax.annotate(
                str(int(vote_row["top_n"])),
                (vote_row["spearman_rho"], vote_row["auc_discrimination"]),
                xytext=(3, 3), textcoords="offset points", fontsize=6, color=color, zorder=10,
            )

    # LLR: plotted as points (Spearman rho vs the DMS functional score) and kept as AUC guide lines.
    color = method_color("LLR baseline")
    for row in llr_rows_for_cutoff(scatter_df, min_stars, scores):
        y = row["auc_discrimination"]
        ax.axhline(y, color=color, linestyle=":", linewidth=1.2, alpha=0.8, zorder=0)
        if np.isfinite(row["spearman_rho"]):
            ax.scatter(
                row["spearman_rho"], y,
                s=115, marker=method_marker("LLR baseline"), color=color,
                alpha=0.96, edgecolor="black", linewidth=0.8,
                label=f"{row['label']} ({y:.3f})", zorder=7,
            )
        else:
            ax.text(0.99, y, f"{row['label']} AUC*={y:.3f}", transform=ax.get_yaxis_transform(), va="bottom", ha="right", fontsize=7, color=color)

    reference_df = panel_df[panel_df["method_family"].isin(REFERENCE_METHODS)].copy()
    reference_df = (
        reference_df[np.isfinite(reference_df["auc_discrimination"])]
        .sort_values("auc_discrimination", ascending=False)
        .drop_duplicates("method_family")
    )
    for _, row in reference_df.iterrows():
        family = row["method_family"]
        y = row["auc_discrimination"]
        ax.axhline(y, color=method_color(family), linestyle="--", linewidth=1.3, alpha=0.8, zorder=0)
        ax.text(0.01, y, f"{method_label(family)} AUC*={y:.3f}", transform=ax.get_yaxis_transform(), va="bottom", ha="left", fontsize=7, color=method_color(family))

    ax.axhline(0.5, color="0.72", linewidth=1, linestyle="--", zorder=1)
    n_benign = int(safe_numeric(scatter_df["n_benign"]).max()) if "n_benign" in scatter_df.columns else 0
    n_pathogenic = int(safe_numeric(scatter_df["n_pathogenic"]).max()) if "n_pathogenic" in scatter_df.columns else 0
    ax.legend(frameon=False, fontsize=7, loc="lower right")
    return n_benign, n_pathogenic

if auc_by_labels.empty:
    print("No AUC rows to plot. Run the metrics/AUC recompute cells first.")
else:
    target_scores = spearman_target_scores()
    fig, axes = plt.subplots(2, 2, figsize=(15.0, 11.0), sharex=True, sharey=True)
    for ax, min_stars in zip(axes.ravel(), GRID_STAR_CUTOFFS):
        n_benign, n_pathogenic = draw_spearman_auc_panel(ax, min_stars, target_scores)
        cutoff_label = "all binary labels" if min_stars == 0 else f">={min_stars} review stars"
        ax.set_title(f"{cutoff_label} (n={n_benign} benign, {n_pathogenic} pathogenic)", fontsize=11)
        ax.set_ylim(0.45, 1.02)

    for ax in axes[-1]:
        ax.set_xlabel("Spearman rho vs BRCA1 functional score")
    for ax in axes[:, 0]:
        ax.set_ylabel("Direction-normalized ClinVar AUC")

    fig.suptitle(f"BRCA1 layers, baselines, and voting ensembles by ClinVar review-star cutoff ({CLINICAL_LABEL_MODE})", y=1.0, fontsize=13)
    fig.tight_layout()
    out = FIGURE_DIR / f"{DATASET}_all_layers_spearman_vs_auc_by_review_stars_{CLINICAL_LABEL_MODE}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(out)

    # Ranked summary directly under the star-cutoff plot. Keep one best row from each
    # non-voting family, then compare those representatives with every voting ensemble.
    star_table_rows = []
    for min_stars in GRID_STAR_CUTOFFS:
        panel = auc_by_labels[auc_by_labels["min_review_stars"].eq(min_stars)].copy()
        panel["spearman_rho"] = safe_numeric(panel.get("spearman_rho", np.nan))
        panel["auc_discrimination"] = safe_numeric(panel["auc_discrimination"])
        panel = panel[np.isfinite(panel["auc_discrimination"])]
        for family, family_rows in panel.groupby("method_family", sort=False):
            row = family_rows.sort_values(["auc_discrimination", "spearman_rho"], ascending=[False, False]).iloc[0]
            detail = str(row.get("model_label", "")).replace("nan", "")
            star_table_rows.append({
                "min_review_stars": min_stars,
                "method": f"{method_label(family)}: {detail}".rstrip(": "),
                "method_group": method_label(family),
                "spearman_rho": row.get("spearman_rho", np.nan),
                "auc_discrimination": row["auc_discrimination"],
                "n_benign": row.get("n_benign", np.nan),
                "n_pathogenic": row.get("n_pathogenic", np.nan),
            })
        for row in llr_rows_for_cutoff(panel, min_stars, target_scores):
            star_table_rows.append({
                "min_review_stars": min_stars, "method": row["label"], "method_group": "LLR",
                "spearman_rho": row["spearman_rho"], "auc_discrimination": row["auc_discrimination"],
                "n_benign": panel["n_benign"].max(), "n_pathogenic": panel["n_pathogenic"].max(),
            })
        voting_rows = voting_ensemble_metrics[voting_ensemble_metrics["min_review_stars"].eq(min_stars)]
        for _, row in voting_rows.iterrows():
            star_table_rows.append({
                "min_review_stars": min_stars, "method": row["ensemble_label"], "method_group": "Voting ensemble",
                "spearman_rho": row["spearman_rho"], "auc_discrimination": row["auc_discrimination"],
                "n_benign": row["n_benign"], "n_pathogenic": row["n_pathogenic"],
            })

    star_plot_best_methods = pd.DataFrame(star_table_rows)
    star_plot_best_methods = (
        star_plot_best_methods.sort_values(
            ["min_review_stars", "auc_discrimination", "spearman_rho"],
            ascending=[True, False, False],
        )
        .groupby("min_review_stars", group_keys=False)
        .head(10)
        .copy()
    )
    star_plot_best_methods["rank"] = star_plot_best_methods.groupby("min_review_stars").cumcount() + 1
    star_plot_best_methods["review_cutoff"] = star_plot_best_methods["min_review_stars"].map(
        lambda value: "all binary" if value == 0 else f">={value} stars"
    )
    star_plot_best_methods = star_plot_best_methods[
        ["review_cutoff", "rank", "method", "method_group", "spearman_rho", "auc_discrimination", "n_benign", "n_pathogenic"]
    ]
    display(star_plot_best_methods)


## Plot: Best Voting Ensemble Classification By Clinical Annotation And Star Level

The best voting ensemble is selected by all-binary direction-normalized AUC (breaking ties by Spearman rho). Its fixed pathogenic vote fractions are then split into benign and pathogenic ClinVar groups at each review-star cutoff, matching the earlier best-method distribution plot.


In [ ]:
plt, sns = ensure_plotting()

all_binary_voting = voting_ensemble_metrics[
    voting_ensemble_metrics["min_review_stars"].eq(0)
    & np.isfinite(safe_numeric(voting_ensemble_metrics["auc_discrimination"]))
].copy()
all_binary_voting["spearman_rho"] = safe_numeric(all_binary_voting["spearman_rho"])
all_binary_voting["auc_discrimination"] = safe_numeric(all_binary_voting["auc_discrimination"])

if all_binary_voting.empty:
    print("No voting ensemble metrics available. Run the voting-ensemble cell first.")
else:
    best_voting_row = all_binary_voting.sort_values(
        ["auc_discrimination", "spearman_rho"], ascending=[False, False]
    ).iloc[0]
    best_voting_label = best_voting_row["ensemble_label"]
    best_voting_predictions = voting_ensemble_predictions[
        voting_ensemble_predictions["ensemble_label"].eq(best_voting_label)
    ][["SequenceIndex", "vote_fraction", "n_votes"]].copy()
    best_voting_predictions["SequenceIndex"] = best_voting_predictions["SequenceIndex"].astype(str)
    total_votes = int(best_voting_predictions["n_votes"].mode().iloc[0])

    label_table = normalize_label_columns(active_labels.copy())
    label_table["SequenceIndex"] = label_table["SequenceIndex"].astype(str)
    panels = []
    for min_stars in GRID_STAR_CUTOFFS:
        cutoff_labels = label_table[
            label_table["stars"].ge(min_stars)
            & label_table["annotation"].isin(PATHOGENICITY_LABELS)
        ][["SequenceIndex", "annotation", "stars"]]
        panel = best_voting_predictions.merge(cutoff_labels, on="SequenceIndex", how="inner")
        if panel.empty:
            continue
        evaluation = panel.copy()
        evaluation["fitness"] = -evaluation["vote_fraction"]
        panels.append((min_stars, panel, auc_from_labeled_fitness(evaluation)))

    print(
        f"Best voting ensemble: {best_voting_label} | votes={total_votes} | "
        f"all-binary AUC*={best_voting_row['auc_discrimination']:.3f} | "
        f"rho={best_voting_row['spearman_rho']:.3f}"
    )
    if not panels:
        print("No labeled variants overlap the best voting ensemble predictions.")
    else:
        fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, squeeze=False)
        palette = {"benign": "#2C7FB8", "pathogenic": "#D7301F"}
        vote_bin_edges = (np.arange(total_votes + 2) - 0.5) / total_votes
        for ax, (min_stars, panel, stats) in zip(axes.ravel(), panels):
            for annotation in ["benign", "pathogenic"]:
                values = panel.loc[panel["annotation"].eq(annotation), "vote_fraction"].to_numpy(dtype=float)
                ax.hist(
                    values, bins=vote_bin_edges, alpha=0.62, color=palette[annotation],
                    label=f"{annotation} (n={len(values)})", edgecolor="white", linewidth=0.5,
                )
            ax.axvline(0.5, color="0.35", linestyle="--", linewidth=1.1, label="majority-vote boundary")
            cutoff_label = "all binary" if min_stars == 0 else f">={min_stars} stars"
            ax.set_title(f"{cutoff_label} | AUC*={stats['auc_discrimination']:.3f}")
            ax.set_xlabel("Mean pathogenic classification (vote fraction)")
            ax.set_ylabel("Variant count")
            ax.set_xticks(np.arange(total_votes + 1) / total_votes)
            ax.legend(frameon=False, fontsize=8)
        for ax in axes.ravel()[len(panels):]:
            ax.axis("off")
        fig.suptitle(f"BRCA1 best voting ensemble by annotation: {best_voting_label}", y=1.01)
        fig.tight_layout()
        out = FIGURE_DIR / f"{DATASET}_best_voting_ensemble_classification_by_review_stars_{CLINICAL_LABEL_MODE}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(out)
